<a href="https://colab.research.google.com/github/kxenopoulou/epameinondas_xenopoulos_epistemology-of-logic_genetic-historical-logic/blob/main/Poe_com_ANALYSIS_WITH_EPAMEINONDAS_XENOPOULOS_DIALECTICAL_TRANSFORMER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# EPAMEINONDAS XENOPOULOS DIALECTICAL TRANSFORMER (EXDT) v3.0
# ΜΕ XENOPOULOSUNITYFIELD - ΠΛΗΡΗΣ ΣΥΓΧΩΝΕΥΣΗ LAYER & SYSTEM
# ============================================================================

# ----------------------------------------------------------------------------
# ΜΕΡΟΣ 0: IMPORTS
# ----------------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import uuid
import os
import sys
import json
import hashlib
import re
import math
import random
from collections import defaultdict
import getpass

# Colab/Jupyter
try:
    from IPython.display import display, HTML, clear_output
    import ipywidgets as widgets
    IN_COLAB = True
    IPYWIDGETS_AVAILABLE = True
except ImportError:
    IN_COLAB = False
    IPYWIDGETS_AVAILABLE = False
    display = print
    HTML = lambda x: x
    clear_output = lambda: None

# ============================================================================
# ΜΕΡΟΣ 1: ΣΤΑΘΕΡΕΣ ΚΑΙ ΠΑΡΑΜΕΤΡΟΙ
# ============================================================================

VERSION = "EXDT-3.0-UNITY"
CREATOR = "Epameinondas Xenopoulos"
BIRTH_DATE = "1920-08-15"

# Χρονικά κατώφλια
TIME_THRESHOLDS = {i: max(0.45, 0.75 - i*0.01) for i in range(1, 35)}
TIME_THRESHOLDS.update({i: 0.75 for i in range(1, 6)})

# Παράμετροι
P = {
    'α1': 0.35, 'α2': 0.28, 'α3': 0.42, 'α4': 0.19,
    'α5': 0.75, 'α6': 0.63, 'α7': 0.81, 'α8': 0.57,
    'α9': 0.44, 'α10': 0.92, 'α11': 0.38, 'α12': 0.73,
    'β1': 0.25, 'γ1': 0.15, 'δ1': 0.30, 'ε1': 0.20,
    'ζ1': 0.40, 'η1': 0.18, 'θ1': 0.22, 'ι1': 0.28,
    'κ1': 0.32, 'λ1': 0.38, 'μ1': 0.42, 'ν1': 0.48,
    'ξ1': 0.52, 'ο1': 0.58, 'π1': 0.62, 'ρ1': 0.68,
    'σ1': 0.72, 'τ1': 0.78, 'υ1': 0.82, 'φ1': 0.88,
    'χ1': 0.92, 'ψ1': 0.96, 'ω1': 0.99
}

# Στάδια διαλεκτικής εξέλιξης
STAGES = {
    0: {"name": "τ₀: ΣΥΝΟΧΗ", "color": "#2E8B57", "symbol": "✅", "min": 0.0, "max": 0.15},
    1: {"name": "τ₁: ΠΡΩΤΗ ΑΝΩΜΑΛΙΑ", "color": "#3CB371", "symbol": "⚠️", "min": 0.15, "max": 0.30},
    2: {"name": "τ₂: ΕΠΑΝΑΛΗΨΗ ΑΝΩΜΑΛΙΑΣ", "color": "#FFD700", "symbol": "🔄", "min": 0.30, "max": 0.45},
    3: {"name": "τ₃: ΣΥΣΣΩΡΕΥΣΗ ΕΝΤΑΣΗΣ", "color": "#FFA500", "symbol": "📈", "min": 0.45, "max": 0.60},
    4: {"name": "τ₄: ΣΗΜΕΙΟ ΚΡΙΣΗΣ", "color": "#FF6346", "symbol": "⚡", "min": 0.60, "max": 0.75},
    5: {"name": "τ₅: ΠΟΙΟΤΙΚΟ ΑΛΜΑ", "color": "#DC143C", "symbol": "⤊", "min": 0.75, "max": 0.85},
    6: {"name": "τ₆: ΠΑΡΑΔΟΞΟ", "color": "#8A2BE2", "symbol": "⟡", "min": 0.85, "max": 0.95},
    7: {"name": "τ₇: ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ", "color": "#483D8B", "symbol": "🌀", "min": 0.0, "max": 0.25},
    8: {"name": "τ₈: ΜΕΤΑ-ΣΤΑΘΕΡΟΤΗΤΑ", "color": "#000080", "symbol": "∞", "min": 0.95, "max": 1.5},
    9: {"name": "τ₉: ΜΕΤΑ-ΥΠΕΡΒΑΣΗ", "color": "#000000", "symbol": "🌟", "min": 1.5, "max": float('inf')}
}

# ============================================================================
# ΜΕΡΟΣ 2: XENOPOULOSUNITYFIELD - ΤΟ ΕΝΟΠΟΙΗΜΕΝΟ ΠΕΔΙΟ
# ============================================================================

class XenopoulosUnityField:
    """
    Το Ενοποιημένο Πεδίο του Ξενόπουλου (XenopoulosUnityField)

    Χ(x, t) = L(x) ⊕ ∫₀ᵗ S(τ) dτ

    όπου:
    - L(x): το στιγμιαίο layer (Αρχή 4: Υπέρβαση)
    - S(τ): το ιστορικό σύστημα (Αρχή 6: Αυτο-αναφορικότητα)
    - ⊕: τελεστής διαλεκτικής σύνθεσης (Αρχή 34: Ολοκλήρωση-Υπέρβαση)
    - Ανίχνευση παραδόξων (Αρχή 9: Παραδοξογένεση)
    """

    def __init__(self, w=0.5, r=0.15, horizon=100):
        self.w = w
        self.r = r
        self.horizon = horizon
        self.history = []          # Ιστορία τιμών Α
        self.field_history = []     # Ιστορία τιμών πεδίου
        self.time = 0

    # ------------------------------------------------------------------------
    # 1. ΣΤΙΓΜΙΑΙΟ ΜΕΡΟΣ (Layer) - Αρχή 4: Υπέρβαση
    # ------------------------------------------------------------------------
    def layer(self, x):
        """
        L(x) = tanh(x·w)·(1 - r)

        Το στιγμιαίο μέρος - η άμεση απόκριση στο ερέθισμα.
        """
        return np.tanh(x * self.w) * (1 - self.r)

    # ------------------------------------------------------------------------
    # 2. ΙΣΤΟΡΙΚΟ ΜΕΡΟΣ (System) - Αρχή 6: Αυτο-αναφορικότητα
    # ------------------------------------------------------------------------
    def system(self, t, A):
        """
        S(t) = ¬ᴰ(A) = -A·p·h·(1 + m) + ε

        Το ιστορικό μέρος - επηρεάζεται από το παρελθόν.
        """
        # Memory effect (αυτο-αναφορικότητα)
        if len(self.history) > 0:
            window = min(10, len(self.history))
            recent = np.mean(self.history[-window:])
            memory = 0.2 * np.tanh(recent * 2)
        else:
            memory = 0

        # Στοχαστικοί παράγοντες
        p = 0.7 + 0.3 * random.random()      # preservation
        h = 1.0 + 0.3 * random.random()      # historical weight
        noise = 0.05 * (1 + abs(A)) * random.gauss(0, 1)

        # Διαλεκτική άρνηση
        return -A * p * h * (1 + memory) + noise

    # ------------------------------------------------------------------------
    # 3. ΤΕΛΕΣΤΗΣ ΔΙΑΛΕΚΤΙΚΗΣ ΣΥΝΘΕΣΗΣ (⊕) - Αρχή 34: Ολοκλήρωση-Υπέρβαση
    # ------------------------------------------------------------------------
    def compose(self, a, b):
        """
        a ⊕ b = (a + b)·(1 - |tanh(a·b)|) + i·(a·b)

        Ο τελεστής που ενώνει στιγμιαίο και ιστορικό μέρος.
        Το αποτέλεσμα είναι ΜΙΓΑΔΙΚΟ:
        - Re: η σύνθεση (τι ισχυρίζεται το σύστημα)
        - Im: η ένταση (πόσο "ταράζεται" το σύστημα)
        """
        real_part = (a + b) * (1 - abs(np.tanh(a * b + 1e-10)))
        imag_part = a * b
        return real_part + 1j * imag_part

    # ------------------------------------------------------------------------
    # 4. ΥΠΟΛΟΓΙΣΜΟΣ ΕΝΤΑΣΗΣ
    # ------------------------------------------------------------------------
    def tension(self, X):
        """T = |Im(Χ)| - η διαλεκτική ένταση"""
        return abs(np.imag(X))

    # ------------------------------------------------------------------------
    # 5. ΤΑΞΙΝΟΜΗΣΗ ΣΤΑΔΙΟΥ - Αρχή 9: Παραδοξογένεση
    # ------------------------------------------------------------------------
    def classify_stage(self, X):
        """
        Ταξινόμηση σε στάδιο με βάση:
        - Re(Χ): τι ισχυρίζεται
        - Im(Χ): ένταση
        - Tension: κρυμμένη ένταση
        """
        Re = np.real(X)
        Im = np.imag(X)
        T = self.tension(X)

        # Στάδιο 6: ΠΑΡΑΔΟΞΟ (τ₆)
        if abs(Re) > 0.85 and abs(Im) > 0.85 and T < 0.4:
            return 6, STAGES[6]["name"]

        # Στάδιο 7: ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)
        if T < 0.25 and (abs(Re) > 0.75 or abs(Im) > 0.75):
            return 7, STAGES[7]["name"]

        # Κανονικά στάδια με βάση την ένταση
        for stage_id, stage_info in STAGES.items():
            if stage_id not in [6, 7]:  # Εξαίρεση ειδικών σταδίων
                if stage_info["min"] <= T < stage_info["max"]:
                    return stage_id, stage_info["name"]

        return 0, STAGES[0]["name"]

    # ------------------------------------------------------------------------
    # 6. ΥΠΟΛΟΓΙΣΜΟΣ XEPTQLRI
    # ------------------------------------------------------------------------
    def calculate_xeptqlri(self, X):
        """
        XEPTQLRI = [T^1.2 · trend · paradox · asym] / 0.85
        """
        Re = np.real(X)
        Im = np.imag(X)
        T = self.tension(X)

        # Base
        base = T ** 1.2

        # Trend factor (από ιστορία)
        if len(self.field_history) >= 5:
            recent = [np.imag(f) for f in self.field_history[-5:]]
            if len(recent) > 1:
                trend_coef = np.polyfit(range(len(recent)), recent, 1)[0]
                trend = 1.0 + abs(trend_coef) * 15
            else:
                trend = 1.0
        else:
            trend = 1.0

        # Paradox factor
        paradox = 1.0
        if abs(Re) > 0.85 and abs(Im) > 0.85:
            paradox = 2.8 if T < 0.4 else 2.0

        # Asymmetry factor
        asym = 1.0 + (1 - abs(abs(Re) - abs(Im))) * 0.5

        return min(5.0, max(0.0, (base * trend * paradox * asym) / 0.85))

    # ------------------------------------------------------------------------
    # 7. ΟΛΟΚΛΗΡΩΤΙΚΗ ΜΟΡΦΗ - ΤΟ ΠΛΗΡΕΣ ΠΕΔΙΟ
    # ------------------------------------------------------------------------
    def forward(self, x, A=None):
        """
        Χ(x, t) = L(x) ⊕ ∫₀ᵗ S(τ) dτ

        Υπολογίζει το πεδίο τη χρονική στιγμή t.
        """
        # Αν δεν δόθηκε Α, υπολόγισε από ιστορία
        if A is None:
            A = 0.5 + 0.3 * np.sin(self.time * 0.1) if len(self.history) > 0 else 0.5

        # Στιγμιαίο μέρος
        L = self.layer(x)

        # Ολοκλήρωμα ιστορικού μέρους
        integral = 0
        for tau in range(min(self.time, self.horizon)):
            if tau < len(self.history):
                integral += self.system(tau, self.history[tau])

        # Σύνθεση
        X = self.compose(L, integral)

        # Καταγραφή
        self.field_history.append(X)
        self.history.append(A)
        self.time += 1

        # Υπολογισμός μετρικών
        stage_id, stage_name = self.classify_stage(X)
        xeptqlri = self.calculate_xeptqlri(X)

        return {
            'field': X,
            'real': np.real(X),
            'imag': np.imag(X),
            'tension': self.tension(X),
            'stage_id': stage_id,
            'stage': stage_name,
            'xeptqlri': xeptqlri,
            'layer': L,
            'integral': integral,
            'time': self.time
        }

    # ------------------------------------------------------------------------
    # 8. ΕΝΤΟΠΙΣΜΟΣ ΠΑΡΑΔΟΞΩΝ ΣΕ ΚΕΙΜΕΝΟ
    # ------------------------------------------------------------------------
    def analyze_text(self, text):
        """
        Αναλύει ένα κείμενο για παράδοξα και αντιφάσεις.
        """
        # Χωρισμός σε προτάσεις
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        results = []

        for i, sentence in enumerate(sentences):
            # Υπολογισμός φορτίου πρότασης
            words = sentence.lower().split()

            # Ανίχνευση συναισθηματικού φορτίου
            positive_words = ['ευγνωμοσύνη', 'εμπιστοσύνη', 'χαρά', 'καλός', 'σωστός']
            negative_words = ['μηνύω', 'καταγγέλλω', 'ψέμα', 'λάθος', 'πρόβλημα']

            pos_count = sum(1 for w in words if w in positive_words)
            neg_count = sum(1 for w in words if w in negative_words)

            # Ανίχνευση χρονολογιών
            dates = re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', sentence)

            # Ανίχνευση ποσών
            amounts = re.findall(r'\d+[.,]?\d*\s*[€$]', sentence)

            # Υπολογισμός εισόδου για το πεδίο
            x = (pos_count - neg_count) / max(len(words), 1) + 0.5

            # Εφαρμογή πεδίου
            result = self.forward(x)
            results.append({
                'sentence': sentence[:50] + ('...' if len(sentence) > 50 else ''),
                'index': i,
                'field': result,
                'dates': dates,
                'amounts': amounts,
                'pos_neg': (pos_count, neg_count)
            })

        return results

    # ------------------------------------------------------------------------
    # 9. ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ
    # ------------------------------------------------------------------------
    def statistics(self):
        """Στατιστικά του πεδίου"""
        if not self.field_history:
            return {}

        real_parts = [np.real(f) for f in self.field_history]
        imag_parts = [np.imag(f) for f in self.field_history]

        return {
            'real_mean': float(np.mean(real_parts)),
            'real_std': float(np.std(real_parts)),
            'real_max': float(np.max(real_parts)),
            'real_min': float(np.min(real_parts)),
            'imag_mean': float(np.mean(imag_parts)),
            'imag_std': float(np.std(imag_parts)),
            'imag_max': float(np.max(imag_parts)),
            'imag_min': float(np.min(imag_parts)),
            'energy': float(np.sum([abs(f)**2 for f in self.field_history])),
            'steps': len(self.field_history)
        }

    # ------------------------------------------------------------------------
    # 10. RESET
    # ------------------------------------------------------------------------
    def reset(self):
        """Επαναφορά πεδίου"""
        self.history = []
        self.field_history = []
        self.time = 0


# ============================================================================
# ΜΕΡΟΣ 3: TEXT ANALYZER (Προσαρμοσμένος για UnityField)
# ============================================================================

class TextAnalyzer:
    """Ανάλυση κειμένου με UnityField"""

    def __init__(self):
        self.field = XenopoulosUnityField()
        self.analyses = []

    def lexical_contradictions(self, text):
        """Λεξικές αντιφάσεις (αλλά, όμως, κλπ)"""
        words = ['αλλά', 'όμως', 'παρόλα', 'αντίθετα', 'μολονότι', 'ωστόσο', 'ενώ']
        count = 0
        sentences = [s for s in text.split('.') if s.strip()]

        for i, s in enumerate(sentences):
            found = [w for w in words if w in s.lower()]
            if found:
                count += 1

        return count

    def semantic_analysis(self, text):
        """Νοηματική ανάλυση με UnityField"""
        self.field.reset()
        results = self.field.analyze_text(text)

        # Συγκεντρωτικά στατιστικά
        real_vals = [r['field']['real'] for r in results]
        imag_vals = [r['field']['imag'] for r in results]
        tensions = [r['field']['tension'] for r in results]
        stages = [r['field']['stage_id'] for r in results]

        # Ανίχνευση παραδόξων
        paradoxes = sum(1 for s in stages if s == 6)
        false_stability = sum(1 for s in stages if s == 7)

        return {
            'words': len(text.split()),
            'sentences': len(results),
            'unique_words': len(set(text.lower().split())),
            'lexical_diversity': len(set(text.lower().split())) / max(len(text.split()), 1),
            'avg_sentence_length': len(text.split()) / max(len(results), 1),
            'readability': max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)),
            'lexical_contradictions': self.lexical_contradictions(text),
            'semantic_contradictions': paradoxes + false_stability,
            'paradoxes': paradoxes,
            'false_stability': false_stability,
            'avg_real': float(np.mean(real_vals)) if real_vals else 0,
            'avg_imag': float(np.mean(imag_vals)) if imag_vals else 0,
            'avg_tension': float(np.mean(tensions)) if tensions else 0,
            'max_tension': float(np.max(tensions)) if tensions else 0,
            'score': float(0.3 * (len(set(text.lower().split()))/max(len(text.split()),1)) +
                          0.4 * max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)) +
                          0.3 * (1 - (paradoxes + false_stability)/max(len(results),1))),
            'quality': self._get_quality(0.3 * (len(set(text.lower().split()))/max(len(text.split()),1)) +
                                         0.4 * max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)) +
                                         0.3 * (1 - (paradoxes + false_stability)/max(len(results),1)))
        }

    def _get_quality(self, score):
        if score > 0.8: return "ΕΞΑΙΡΕΤΙΚΗ"
        if score > 0.6: return "ΚΑΛΗ"
        if score > 0.4: return "ΜΕΤΡΙΑ"
        return "ΧΑΜΗΛΗ"

    def full_metrics(self, text):
        """Πλήρεις μετρικές"""
        semantic = self.semantic_analysis(text)

        # Υπολογισμός XEPTQLRI από τελευταία πρόταση
        self.field.reset()
        results = self.field.analyze_text(text)
        last_xeptqlri = results[-1]['field']['xeptqlri'] if results else 0

        return {
            'Βασικά': {
                'Χαρακτήρες': len(text),
                'Λέξεις': semantic['words'],
                'Προτάσεις': semantic['sentences']
            },
            'Γλωσσικά': {
                'Λεξιλογική ποικιλία': f"{semantic['lexical_diversity']*100:.1f}%",
                'Μ.Ο. μήκος πρότασης': f"{semantic['avg_sentence_length']:.1f}",
                'Αναγνωσιμότητα': f"{semantic['readability']*100:.1f}%"
            },
            'Αντιφάσεις': {
                'Λεξικές/Συντακτικές': semantic['lexical_contradictions'],
                'Νοηματικές': semantic['semantic_contradictions'],
                'Παράδοξα (τ₆)': semantic['paradoxes'],
                'Ψευδής σταθερότητα (τ₇)': semantic['false_stability'],
                'Σύνολο': semantic['lexical_contradictions'] + semantic['semantic_contradictions']
            },
            'XEPTQLRI': {
                'Δείκτης': f"{last_xeptqlri:.4f}",
                'Μέση ένταση': f"{semantic['avg_tension']:.4f}",
                'Μέγιστη ένταση': f"{semantic['max_tension']:.4f}"
            },
            'Ποιότητα': semantic['quality']
        }

    def final_assessment(self, text):
        """Τελική αποτίμηση με UnityField"""
        semantic = self.semantic_analysis(text)

        result = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║              ΤΕΛΙΚΗ ΑΠΟΤΙΜΗΣΗ ΜΕ XENOPOULOSUNITYFIELD                    ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  ⚠️  ΠΙΝΑΚΑΣ ΑΝΤΙΦΑΣΕΩΝ                                                  ║
║  ────────────────────────────────────────────────────────────────────── ║
║                                                                          ║
║  Τύπος Αντίφασης                  |  Πλήθος  |  Σοβαρότητα              ║
║  ────────────────────────────────────────────────────────────────────── ║
║  Λεξικές/Συντακτικές                |  {semantic['lexical_contradictions']:4d}      |  25%                       ║
║  Νοηματικές (σύνολο)                |  {semantic['semantic_contradictions']:4d}      |  Ποικίλη                   ║
║    • Παράδοξα (τ₆)                   |  {semantic['paradoxes']:4d}      |  85% (ΚΡΙΣΙΜΟ)              ║
║    • Ψευδής Σταθερότητα (τ₇)         |  {semantic['false_stability']:4d}      |  70% (ΥΠΟΠΤΟ)               ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  🔴 ΑΝΑΛΥΣΗ ΠΕΔΙΟΥ:                                                      ║
║     • Μέσο Re(Χ): {semantic['avg_real']:.4f} (τι ισχυρίζεται)                           ║
║     • Μέσο Im(Χ): {semantic['avg_imag']:.4f} (διαλεκτική ένταση)                        ║
║     • Μέση ένταση: {semantic['avg_tension']:.4f}                                        ║
║     • Μέγιστη ένταση: {semantic['max_tension']:.4f}                                      ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  🔍 ΕΝΤΟΠΙΣΜΕΝΑ ΠΑΡΑΔΟΞΑ:                                                ║
"""

        if semantic['paradoxes'] > 0:
            result += f"""
║     • Βρέθηκαν {semantic['paradoxes']} ΠΑΡΑΔΟΞΑ (τ₆) - ΑΜΕΣΗ ΠΡΟΣΟΧΗ!    ║
║     • Το κείμενο περιέχει ΛΟΓΙΚΕΣ ΑΝΤΙΦΑΣΕΙΣ                             ║
"""
        else:
            result += """
║     • Δεν βρέθηκαν παράδοξα (τ₆) - ΚΑΛΟ!                                 ║
"""

        if semantic['false_stability'] > 0:
            result += f"""
║     • Βρέθηκαν {semantic['false_stability']} περιπτώσεις ΨΕΥΔΟΥΣ ΣΤΑΘΕΡΟΤΗΤΑΣ (τ₇)   ║
║     • Υπάρχουν ΚΡΥΦΕΣ αντιφάσεις - ΠΡΟΣΟΧΗ!                              ║
"""

        result += f"""
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  💡 ΣΧΕΔΙΟ ΔΡΑΣΗΣ:                                                       ║
║                                                                          ║
║  ΦΑΣΗ 1 - Άμεσες ενέργειες (Προτεραιότητα ★★★):                         ║
"""

        if semantic['paradoxes'] > 0:
            result += f"""
║     • Διόρθωσε {semantic['paradoxes']} ΠΑΡΑΔΟΞΑ (λογικές αντιφάσεις)     ║
"""
        else:
            result += """
║     • Καμία άμεση ενέργεια για παράδοξα - ΠΡΟΧΩΡΑ                         ║
"""

        result += f"""
║                                                                          ║
║  ΦΑΣΗ 2 - Βελτιστοποίηση (Προτεραιότητα ✦):                             ║
║     • Διόρθωσε {semantic['lexical_contradictions']} λεξικές/συντακτικές αντιφάσεις   ║
║     • Διόρθωσε {semantic['false_stability']} περιπτώσεις ψευδούς σταθερότητας       ║
║                                                                          ║
║  ΦΑΣΗ 3 - Τελικός έλεγχος:                                               ║
║     • Επαναξιολόγηση με UnityField                                      ║
║     • Σύγκριση αρχικού-διορθωμένου                                       ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
        return result

    def unity_field_report(self, text):
        """Αναλυτική αναφορά UnityField"""
        self.field.reset()
        results = self.field.analyze_text(text)

        report = f"""
🌀 XENOPOULOS UNITY FIELD - ΑΝΑΛΥΤΙΚΗ ΑΝΑΦΟΡΑ
================================================================================

📊 ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ:
   • Βήματα: {len(results)}
   • Τελικό Re(Χ): {results[-1]['field']['real']:.4f}
   • Τελικό Im(Χ): {results[-1]['field']['imag']:.4f}
   • Τελική ένταση: {results[-1]['field']['tension']:.4f}
   • Τελικό στάδιο: {results[-1]['field']['stage']}
   • Τελικό XEPTQLRI: {results[-1]['field']['xeptqlri']:.4f}

🔍 ΑΝΑΛΥΣΗ ΑΝΑ ΠΡΟΤΑΣΗ:
"""
        for i, r in enumerate(results[:10]):  # Πρώτες 10 προτάσεις
            report += f"""
   {i+1}. {r['sentence']}
       • Re={r['field']['real']:.3f}, Im={r['field']['imag']:.3f}, T={r['field']['tension']:.3f}
       • Στάδιο: {r['field']['stage']}
       • Ημερομηνίες: {r['dates']}, Ποσά: {r['amounts']}
"""

        if len(results) > 10:
            report += f"\n   ... και {len(results)-10} ακόμη προτάσεις"

        stats = self.field.statistics()
        report += f"""

📈 ΣΥΝΟΛΙΚΑ ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ:
   • Μέσο Re(Χ): {stats.get('real_mean', 0):.4f}
   • Μέσο Im(Χ): {stats.get('imag_mean', 0):.4f}
   • Μέγιστη ένταση: {stats.get('imag_max', 0):.4f}
   • Ολική ενέργεια: {stats.get('energy', 0):.4f}
   • Βήματα: {stats.get('steps', 0)}

🏆 ΤΕΛΙΚΗ ΔΙΑΓΝΩΣΗ:
"""
        last = results[-1]['field']
        if last['stage_id'] == 6:
            report += "🔴 ΤΟ ΚΕΙΜΕΝΟ ΠΕΡΙΕΧΕΙ ΠΑΡΑΔΟΞΑ - ΑΠΑΙΤΕΙΤΑΙ ΔΙΟΡΘΩΣΗ!"
        elif last['stage_id'] == 7:
            report += "⚠️ ΤΟ ΚΕΙΜΕΝΟ ΕΧΕΙ ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ - ΥΠΑΡΧΟΥΝ ΚΡΥΦΕΣ ΑΝΤΙΦΑΣΕΙΣ"
        elif last['tension'] < 0.3:
            report += "✅ ΤΟ ΚΕΙΜΕΝΟ ΕΙΝΑΙ ΛΟΓΙΚΑ ΣΥΝΕΠΕΣ"
        else:
            report += f"📊 ΤΟ ΚΕΙΜΕΝΟ ΒΡΙΣΚΕΤΑΙ ΣΕ ΚΑΤΑΣΤΑΣΗ: {last['stage']}"

        return report


# ============================================================================
# ΜΕΡΟΣ 4: ΑΣΦΑΛΕΙΑ
# ============================================================================

class SecuritySystem:
    def __init__(self):
        self.level = 'user'
        self.keys = ['15081920', '15-08-1920', '15 Αυγούστου 1920']

    def authenticate(self, key):
        if key in self.keys:
            self.level = 'admin'
            return True
        return False


# ============================================================================
# ΜΕΡΟΣ 5: MEMORY MANAGER
# ============================================================================

class MemoryManager:
    def __init__(self, memory_file="exdt_memory.json"):
        self.memory_file = memory_file

    def save(self, data):
        with open(self.memory_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False, default=str)
        return True

    def load(self):
        if os.path.exists(self.memory_file):
            with open(self.memory_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        return None


# ============================================================================
# ΜΕΡΟΣ 6: ΠΛΗΡΕΣ ΣΥΣΤΗΜΑ
# ============================================================================

class XenopoulosSystem:
    def __init__(self, memory_file="exdt_memory.json"):
        self.text = TextAnalyzer()
        self.unity_field = XenopoulosUnityField()
        self.security = SecuritySystem()
        self.memory = MemoryManager(memory_file)
        self.analyses = []
        self.load_memory()

    def load_memory(self):
        data = self.memory.load()
        if data:
            self.analyses = data.get('analyses', [])
            print(f"📂 Φορτώθηκαν {len(self.analyses)} προηγούμενες αναλύσεις")

    def save_memory(self):
        data = {'analyses': self.analyses, 'last_save': datetime.now().isoformat()}
        self.memory.save(data)

    def analyze_text(self, text, save=True):
        result = {
            'id': str(uuid.uuid4())[:8],
            'timestamp': datetime.now().isoformat(),
            'text_preview': text[:100] + ('...' if len(text) > 100 else ''),
            'semantic': self.text.semantic_analysis(text),
            'unity': self.text.unity_field_report(text)
        }

        if save:
            self.analyses.append(result)
            if len(self.analyses) % 5 == 0:
                self.save_memory()

        return result

    def info(self):
        return {
            'version': VERSION,
            'access': self.security.level,
            'analyses': len(self.analyses)
        }


# ============================================================================
# ΜΕΡΟΣ 7: UI
# ============================================================================

class XenopoulosUI:
    def __init__(self, system):
        self.sys = system
        self.text = system.text
        if IPYWIDGETS_AVAILABLE:
            self._build()

    def _build(self):
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #667eea, #764ba2);
             padding: 20px; border-radius: 15px; color: white; text-align: center;">
            <h1>🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ</h1>
            <h2>EXDT v3.0 με XENOPOULOSUNITYFIELD</h2>
            <p>9 Λειτουργίες Ανάλυσης | 34 Αρχές | 7 Θεωρήματα</p>
        </div>
        """))

        self.input = widgets.Textarea(
            value='Επικολλήστε το κείμενό σας εδώ...',
            layout=widgets.Layout(width='100%', height='150px')
        )
        display(self.input)

        # 9 κουμπιά
        btns = []
        btn_configs = [
            ('1️⃣ ΛΕΞΙΚΕΣ', 'primary', self._lexical),
            ('2️⃣ ΝΟΗΜΑΤΙΚΗ', 'success', self._semantic),
            ('3️⃣ ΔΙΟΡΘΩΣΗ', 'info', self._correct),
            ('4️⃣ ΛΙΣΤΑ', 'warning', self._list),
            ('5️⃣ ΜΕΤΡΙΚΕΣ', 'danger', self._metrics),
            ('6️⃣ ΣΥΝΟΛΙΚΗ', 'primary', self._report),
            ('7️⃣ ΤΕΛΙΚΗ v3', 'success', self._final),
            ('8️⃣ EXTREME', 'danger', self._extreme),
            ('0️⃣ UNITY FIELD', 'info', self._unity)
        ]

        for txt, style, handler in btn_configs:
            btn = widgets.Button(description=txt, button_style=style,
                               layout=widgets.Layout(width='160px', margin='2px'))
            btn.on_click(handler)
            btns.append(btn)

        display(widgets.HBox(btns[:5]))
        display(widgets.HBox(btns[5:]))

        # Output
        self.out = widgets.Output(layout=widgets.Layout(width='100%', height='500px', overflow='auto'))
        display(self.out)

    def _get_text(self):
        t = self.input.value
        if t in ['Επικολλήστε το κείμενό σας εδώ...', '']:
            with self.out:
                clear_output()
                print("❌ Παρακαλώ εισάγετε κείμενο")
            return None
        return t

    def _lexical(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            c = self.text.lexical_contradictions(t)
            print(f"🔍 ΛΕΞΙΚΕΣ ΑΝΤΙΦΑΣΕΙΣ: {c}")

    def _semantic(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            m = self.text.semantic_analysis(t)
            print("📊 ΝΟΗΜΑΤΙΚΗ ΑΝΑΛΥΣΗ\n" + "="*50)
            for k, v in m.items():
                if k not in ['avg_real', 'avg_imag', 'avg_tension', 'max_tension', 'paradoxes', 'false_stability']:
                    print(f"{k}: {v}")
            print(f"\n📈 ΣΤΑΤΙΣΤΙΚΑ UNITY FIELD:")
            print(f"   Μέσο Re(Χ): {m['avg_real']:.4f}")
            print(f"   Μέσο Im(Χ): {m['avg_imag']:.4f}")
            print(f"   Μέση ένταση: {m['avg_tension']:.4f}")
            print(f"   Μέγιστη ένταση: {m['max_tension']:.4f}")
            print(f"   Παράδοξα (τ₆): {m['paradoxes']}")
            print(f"   Ψευδής σταθερότητα (τ₇): {m['false_stability']}")

    def _correct(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print("✏️ ΔΙΟΡΘΩΣΗ\n" + "="*50)
            print("Η λειτουργία διόρθωσης θα ενσωματωθεί σύντομα")

    def _list(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            self.text.field.reset()
            results = self.text.field.analyze_text(t)
            print("📋 ΛΕΠΤΟΜΕΡΗΣ ΛΙΣΤΑ ΑΝΑΛΥΣΗΣ\n" + "="*50)
            for i, r in enumerate(results[:10]):
                print(f"\n{i+1}. {r['sentence']}")
                print(f"   Re={r['field']['real']:.3f}, Im={r['field']['imag']:.3f}")
                print(f"   Στάδιο: {r['field']['stage']}")
                if r['dates']:
                    print(f"   Ημερομηνίες: {r['dates']}")
                if r['amounts']:
                    print(f"   Ποσά: {r['amounts']}")

    def _metrics(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            m = self.text.full_metrics(t)
            print("📊 ΠΛΗΡΕΙΣ ΜΕΤΡΙΚΕΣ\n" + "="*50)
            for category, values in m.items():
                print(f"\n{category}:")
                if isinstance(values, dict):
                    for k, v in values.items():
                        print(f"  {k}: {v}")
                else:
                    print(f"  {values}")

    def _report(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.final_assessment(t))

    def _final(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.final_assessment(t))

    def _extreme(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            self.text.field.reset()
            results = self.text.field.analyze_text(t)
            last = results[-1]['field']
            print("🔥 ΑΝΑΛΥΣΗ EXTREME ΚΑΤΑΣΤΑΣΗΣ\n" + "="*50)
            print(f"XEPTQLRI: {last['xeptqlri']:.4f}")
            if last['xeptqlri'] >= 2.0:
                print("🚨 EXTREME ΚΑΤΑΣΤΑΣΗ! (XEPTQLRI > 2.0)")
            elif last['xeptqlri'] >= 1.5:
                print("⚠️ ΥΠΕΡΚΡΙΣΙΜΗ ΚΑΤΑΣΤΑΣΗ")
            elif last['stage_id'] == 6:
                print("🔴 ΠΑΡΑΔΟΞΟ (τ₆)")
            elif last['stage_id'] == 7:
                print("⚠️ ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)")
            else:
                print("✅ ΦΥΣΙΟΛΟΓΙΚΗ ΚΑΤΑΣΤΑΣΗ")

    def _unity(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.unity_field_report(t))


# ============================================================================
# ΜΕΡΟΣ 8: ΕΚΚΙΝΗΣΗ
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ - ΔΙΑΛΕΚΤΙΚΟΣ ΜΕΤΑΣΧΗΜΑΤΙΣΤΗΣ")
    print("="*80)
    print("📌 34 Αρχές | 7 Θεωρήματα | 9 Λειτουργίες | XENOPOULOSUNITYFIELD")
    print("📅 Ημερομηνία γέννησης: 15 Αυγούστου 1920")
    print("🔑 Κλειδί: 15081920 | 15-08-1920 | 15 Αυγούστου 1920")
    print("="*80)

    system = XenopoulosSystem()

    if IPYWIDGETS_AVAILABLE:
        ui = XenopoulosUI(system)
        print("\n✅ Σύστημα έτοιμο - 9 λειτουργίες (ΝΕΟ: 0️⃣ UNITY FIELD)")
    else:
        print("⚠️ Εκτέλεση σε console mode")

    print("\n" + "="*80)
    print("🚀 ΝΕΑ ΛΕΙΤΟΥΡΓΙΑ: XENOPOULOSUNITYFIELD")
    print("   • Ενοποιεί Layer (στιγμιαίο) και System (ιστορικό)")
    print("   • Ανιχνεύει ΠΑΡΑΔΟΞΑ (τ₆) και ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)")
    print("   • Χρησιμοποιεί μιγαδική ανάλυση (Re + i·Im)")
    print("="*80)

    # ============================================================================
# ΑΥΤΟΜΑΤΗ ΑΠΟΘΗΚΕΥΣΗ ΟΛΩΝ ΤΩΝ ΑΠΟΤΕΛΕΣΜΑΤΩΝ ΚΑΙ ΓΡΑΦΗΜΑΤΩΝ
# ============================================================================

import json
import pickle
from datetime import datetime
import os
import matplotlib.pyplot as plt
import numpy as np

class EXDTAutoSave:
    """
    Αυτόματη αποθήκευση ΟΛΩΝ των αποτελεσμάτων και γραφημάτων
    """

    def __init__(self, session_name=None):
        # Δημιουργία μοναδικού ονόματος session
        if session_name is None:
            self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        else:
            self.session_id = session_name

        # Δημιουργία φακέλων
        self.base_dir = f"EXDT_Session_{self.session_id}"
        self.text_dir = f"{self.base_dir}/text_reports"
        self.graphs_dir = f"{self.base_dir}/graphs"
        self.json_dir = f"{self.base_dir}/json_data"
        self.summary_dir = f"{self.base_dir}/summary"

        for dir_path in [self.text_dir, self.graphs_dir, self.json_dir, self.summary_dir]:
            os.makedirs(dir_path, exist_ok=True)

        # Αρχείο καταγραφής session
        self.log_file = f"{self.base_dir}/session_log.txt"
        self._log(f"🔥 NEW EXDT SESSION: {self.session_id}")
        self._log(f"📁 Results saved in: {self.base_dir}")

        # Συλλογή όλων των αποτελεσμάτων
        self.all_unity_reports = []
        self.all_metrics = []
        self.all_extreme = []
        self.graph_count = 0

    def _log(self, message):
        """Εσωτερική καταγραφή"""
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(f"{datetime.now().strftime('%H:%M:%S')} - {message}\n")
        print(message)

    # ------------------------------------------------------------------------
    # 1. ΑΠΟΘΗΚΕΥΣΗ ΓΡΑΦΗΜΑΤΩΝ
    # ------------------------------------------------------------------------

    def save_current_figure(self, name, fig=None, dpi=300, formats=['png', 'pdf', 'svg']):
        """
        Αποθήκευση του τρέχοντος γραφήματος σε πολλαπλές μορφές
        """
        if fig is None:
            fig = plt.gcf()

        timestamp = datetime.now().strftime("%H%M%S")
        self.graph_count += 1

        saved_files = []
        for fmt in formats:
            filename = f"{self.graphs_dir}/{self.graph_count:03d}_{name}_{timestamp}.{fmt}"
            fig.savefig(filename, dpi=dpi, bbox_inches='tight', format=fmt)
            saved_files.append(filename)

        self._log(f"📊 Γράφημα {self.graph_count}: {name} αποθηκεύτηκε σε {len(saved_files)} μορφές")
        return saved_files

    def save_unity_field_plots(self, results, name="unity_field"):
        """
        Αποθήκευση όλων των γραφημάτων του Unity Field
        """
        if not results:
            return

        # Δημιουργία figure με 4 υπογραφήματα
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'XenopoulosUnityField - {name}', fontsize=16)

        steps = range(len(results))
        real_parts = [r['field']['real'] for r in results]
        imag_parts = [r['field']['imag'] for r in results]
        tensions = [r['field']['tension'] for r in results]

        # 1. Re(Χ) και Im(Χ)
        ax1 = axes[0, 0]
        ax1.plot(steps, real_parts, 'b-', label='Re(Χ)', linewidth=2)
        ax1.plot(steps, imag_parts, 'r--', label='Im(Χ)', linewidth=2)
        ax1.axhline(y=0.85, color='purple', linestyle=':', label='Όριο παραδόξου')
        ax1.set_xlabel('Χρονικό Βήμα')
        ax1.set_ylabel('Τιμή')
        ax1.set_title('Πραγματικό και Φανταστικό Μέρος')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2. Διαλεκτική Ένταση
        ax2 = axes[0, 1]
        ax2.plot(steps, tensions, 'purple', linewidth=2)
        ax2.axhline(y=0.4, color='r', linestyle='--', label='Όριο παραδόξου')
        ax2.axhline(y=0.25, color='orange', linestyle='--', label='Όριο ψευδούς σταθ.')
        ax2.fill_between(steps, 0, tensions, alpha=0.3, color='purple')
        ax2.set_xlabel('Χρονικό Βήμα')
        ax2.set_ylabel('Ένταση |Im(Χ)|')
        ax2.set_title('Διαλεκτική Ένταση')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3. Φασικό διάγραμμα
        ax3 = axes[1, 0]
        scatter = ax3.scatter(real_parts, imag_parts, c=steps, cmap='viridis',
                            alpha=0.7, s=50)
        plt.colorbar(scatter, ax=ax3, label='Χρονικό Βήμα')
        ax3.axhline(y=0.85, color='r', linestyle='--', alpha=0.5)
        ax3.axvline(x=0.85, color='r', linestyle='--', alpha=0.5)
        ax3.set_xlabel('Re(Χ)')
        ax3.set_ylabel('Im(Χ)')
        ax3.set_title('Φασικό Διάγραμμα')
        ax3.grid(True, alpha=0.3)

        # 4. Ιστόγραμμα
        ax4 = axes[1, 1]
        ax4.hist(tensions, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
        ax4.axvline(x=0.4, color='r', linestyle='--', label='Όριο παραδόξου')
        ax4.axvline(x=0.25, color='orange', linestyle='--', label='Όριο ψευδούς')
        ax4.set_xlabel('Διαλεκτική Ένταση')
        ax4.set_ylabel('Συχνότητα')
        ax4.set_title('Κατανομή Εντάσεων')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()

        # Αποθήκευση
        return self.save_current_figure(f"unity_field_{name}", fig)

    # ------------------------------------------------------------------------
    # 2. ΑΠΟΘΗΚΕΥΣΗ ΑΝΑΦΟΡΩΝ
    # ------------------------------------------------------------------------

    def save_unity_report(self, report_text, name="unity_report"):
        """Αποθήκευση αναφοράς Unity Field"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.text_dir}/{name}_{timestamp}.txt"

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_text)

        self._log(f"📝 Αναφορά αποθηκεύτηκε: {filename}")

        # Επίσης αποθήκευση σε JSON
        self.save_json({
            'type': 'unity_report',
            'timestamp': datetime.now().isoformat(),
            'report': report_text,
            'report_length': len(report_text)
        }, f"unity_report_{timestamp}")

        self.all_unity_reports.append({
            'filename': filename,
            'timestamp': datetime.now().isoformat()
        })

        return filename

    def save_metrics(self, metrics_dict, name="metrics"):
        """Αποθήκευση μετρικών"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.json_dir}/{name}_{timestamp}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(metrics_dict, f, indent=2, ensure_ascii=False, default=str)

        self._log(f"📊 Μετρικές αποθηκεύτηκαν: {filename}")
        self.all_metrics.append(metrics_dict)

        # Δημιουργία και σύνοψης
        self._create_metrics_summary(metrics_dict, timestamp)

        return filename

    def save_extreme_analysis(self, result, name="extreme"):
        """Αποθήκευση extreme analysis"""

        timestamp = datetime.now().strftime("%H%M%S")

        # Κείμενο
        text_file = f"{self.text_dir}/{name}_{timestamp}.txt"
        with open(text_file, 'w', encoding='utf-8') as f:
            if 'xeptqlri' in result:
                f.write(f"XEPTQLRI: {result['xeptqlri']}\n")
                if result['xeptqlri'] >= 2.0:
                    f.write("🚨 EXTREME ΚΑΤΑΣΤΑΣΗ!\n")

        # JSON
        json_file = f"{self.json_dir}/{name}_{timestamp}.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, default=str)

        self._log(f"🔥 Extreme analysis: {text_file}")
        self.all_extreme.append(result)

        return text_file

    def save_json(self, data, name="data"):
        """Γενική αποθήκευση JSON"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.json_dir}/{name}_{timestamp}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False, default=str)

        return filename

    # ------------------------------------------------------------------------
    # 3. ΣΥΝΟΨΕΙΣ ΚΑΙ ΣΤΑΤΙΣΤΙΚΑ
    # ------------------------------------------------------------------------

    def _create_metrics_summary(self, metrics, timestamp):
        """Δημιουργία σύνοψης από μετρικές"""

        summary_file = f"{self.summary_dir}/summary_{timestamp}.txt"

        with open(summary_file, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write("📊 ΣΥΝΟΨΗ ΜΕΤΡΙΚΩΝ\n")
            f.write("="*60 + "\n\n")

            if 'Βασικά' in metrics:
                f.write("📌 ΒΑΣΙΚΑ ΣΤΟΙΧΕΙΑ:\n")
                for k, v in metrics['Βασικά'].items():
                    f.write(f"   {k}: {v}\n")
                f.write("\n")

            if 'XEPTQLRI' in metrics:
                f.write("🔥 XEPTQLRI:\n")
                for k, v in metrics['XEPTQLRI'].items():
                    f.write(f"   {k}: {v}\n")
                f.write("\n")

            if 'Ποιότητα' in metrics:
                f.write(f"⭐ ΠΟΙΟΤΗΤΑ: {metrics['Ποιότητα']}\n")

        self._log(f"📋 Σύνοψη δημιουργήθηκε: {summary_file}")

    def create_final_report(self):
        """Δημιουργία τελικής αναφοράς όλης της συνεδρίας"""

        report_file = f"{self.base_dir}/FINAL_REPORT.txt"

        with open(report_file, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("🏛️ EXDT v3.0 - ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ ΣΥΝΕΔΡΙΑΣ\n")
            f.write(f"📅 Session ID: {self.session_id}\n")
            f.write(f"🕒 Ημερομηνία: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")

            f.write("📊 ΣΤΑΤΙΣΤΙΚΑ ΣΥΝΕΔΡΙΑΣ:\n")
            f.write(f"   • Unity Reports: {len(self.all_unity_reports)}\n")
            f.write(f"   • Μετρικές: {len(self.all_metrics)}\n")
            f.write(f"   • Extreme Analyses: {len(self.all_extreme)}\n")
            f.write(f"   • Γραφήματα: {self.graph_count}\n\n")

            if self.all_metrics:
                f.write("🔥 ΜΕΓΙΣΤΕΣ ΕΝΤΑΣΕΙΣ:\n")
                for i, m in enumerate(self.all_metrics[-5:]):  # Τελευταίες 5
                    if 'XEPTQLRI' in m:
                        intensity = m['XEPTQLRI'].get('Μέγιστη ένταση', 'N/A')
                        f.write(f"   • Ανάλυση {i+1}: {intensity}\n")

            f.write("\n" + "="*80 + "\n")
            f.write("📁 ΟΛΑ ΤΑ ΑΡΧΕΙΑ ΑΠΟΘΗΚΕΥΤΗΚΑΝ\n")
            f.write(f"📂 Φάκελος: {self.base_dir}\n")

        self._log(f"\n🎉 ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ: {report_file}")
        self._log(f"📁 ΟΛΑ ΤΑ ΑΡΧΕΙΑ ΣΤΟΝ ΦΑΚΕΛΟ: {self.base_dir}")

        return report_file

    # ------------------------------------------------------------------------
    # 4. ΒΟΗΘΗΤΙΚΕΣ ΣΥΝΑΡΤΗΣΕΙΣ
    # ------------------------------------------------------------------------

    def zip_session(self):
        """Δημιουργία zip αρχείου με όλη τη συνεδρία"""

        import shutil

        zip_filename = f"{self.base_dir}.zip"
        shutil.make_archive(self.base_dir, 'zip', self.base_dir)

        self._log(f"📦 Συνεδρία συμπιέστηκε: {zip_filename}")
        return zip_filename

    def download_session(self):
        """Αυτόματο download του zip στο Colab"""

        try:
            from google.colab import files
            zip_file = self.zip_session()
            files.download(zip_file)
            self._log("📥 Το αρχείο κατέβηκε στον υπολογιστή σου!")
        except:
            self._log("⚠️ Δεν μπορεί να γίνει αυτόματο download (μη-Colab περιβάλλον)")


# ============================================================================
# ΠΑΡΑΔΕΙΓΜΑ ΧΡΗΣΗΣ
# ============================================================================

# Δημιουργία αυτόματης αποθήκευσης
saver = EXDTAutoSave("DeepSeek_Analysis_19_3_2026")

# ΜΕΤΑ ΑΠΟ ΚΑΘΕ ΑΝΑΛΥΣΗ:

# 1. Για αποθήκευση αναφοράς Unity Field:
# saver.save_unity_report(unity_report_text, "deepseek_analysis_1")

# 2. Για αποθήκευση γραφημάτων:
# saver.save_unity_field_plots(results, "deepseek_analysis")

# 3. Για αποθήκευση μετρικών:
# saver.save_metrics(metrics_dict, "deepseek_metrics")

# 4. ΣΤΟ ΤΕΛΟΣ:
# saver.create_final_report()
# saver.download_session()  # Κατεβάζει όλα τα αρχεία!# ============================================================================
# EPAMEINONDAS XENOPOULOS DIALECTICAL TRANSFORMER (EXDT) v3.0
# ΜΕ XENOPOULOSUNITYFIELD - ΠΛΗΡΗΣ ΣΥΓΧΩΝΕΥΣΗ LAYER & SYSTEM
# ============================================================================

# ----------------------------------------------------------------------------
# ΜΕΡΟΣ 0: IMPORTS
# ----------------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import uuid
import os
import sys
import json
import hashlib
import re
import math
import random
from collections import defaultdict
import getpass

# Colab/Jupyter
try:
    from IPython.display import display, HTML, clear_output
    import ipywidgets as widgets
    IN_COLAB = True
    IPYWIDGETS_AVAILABLE = True
except ImportError:
    IN_COLAB = False
    IPYWIDGETS_AVAILABLE = False
    display = print
    HTML = lambda x: x
    clear_output = lambda: None

# ============================================================================
# ΜΕΡΟΣ 1: ΣΤΑΘΕΡΕΣ ΚΑΙ ΠΑΡΑΜΕΤΡΟΙ
# ============================================================================

VERSION = "EXDT-3.0-UNITY"
CREATOR = "Epameinondas Xenopoulos"
BIRTH_DATE = "1920-08-15"

# Χρονικά κατώφλια
TIME_THRESHOLDS = {i: max(0.45, 0.75 - i*0.01) for i in range(1, 35)}
TIME_THRESHOLDS.update({i: 0.75 for i in range(1, 6)})

# Παράμετροι
P = {
    'α1': 0.35, 'α2': 0.28, 'α3': 0.42, 'α4': 0.19,
    'α5': 0.75, 'α6': 0.63, 'α7': 0.81, 'α8': 0.57,
    'α9': 0.44, 'α10': 0.92, 'α11': 0.38, 'α12': 0.73,
    'β1': 0.25, 'γ1': 0.15, 'δ1': 0.30, 'ε1': 0.20,
    'ζ1': 0.40, 'η1': 0.18, 'θ1': 0.22, 'ι1': 0.28,
    'κ1': 0.32, 'λ1': 0.38, 'μ1': 0.42, 'ν1': 0.48,
    'ξ1': 0.52, 'ο1': 0.58, 'π1': 0.62, 'ρ1': 0.68,
    'σ1': 0.72, 'τ1': 0.78, 'υ1': 0.82, 'φ1': 0.88,
    'χ1': 0.92, 'ψ1': 0.96, 'ω1': 0.99
}

# Στάδια διαλεκτικής εξέλιξης
STAGES = {
    0: {"name": "τ₀: ΣΥΝΟΧΗ", "color": "#2E8B57", "symbol": "✅", "min": 0.0, "max": 0.15},
    1: {"name": "τ₁: ΠΡΩΤΗ ΑΝΩΜΑΛΙΑ", "color": "#3CB371", "symbol": "⚠️", "min": 0.15, "max": 0.30},
    2: {"name": "τ₂: ΕΠΑΝΑΛΗΨΗ ΑΝΩΜΑΛΙΑΣ", "color": "#FFD700", "symbol": "🔄", "min": 0.30, "max": 0.45},
    3: {"name": "τ₃: ΣΥΣΣΩΡΕΥΣΗ ΕΝΤΑΣΗΣ", "color": "#FFA500", "symbol": "📈", "min": 0.45, "max": 0.60},
    4: {"name": "τ₄: ΣΗΜΕΙΟ ΚΡΙΣΗΣ", "color": "#FF6346", "symbol": "⚡", "min": 0.60, "max": 0.75},
    5: {"name": "τ₅: ΠΟΙΟΤΙΚΟ ΑΛΜΑ", "color": "#DC143C", "symbol": "⤊", "min": 0.75, "max": 0.85},
    6: {"name": "τ₆: ΠΑΡΑΔΟΞΟ", "color": "#8A2BE2", "symbol": "⟡", "min": 0.85, "max": 0.95},
    7: {"name": "τ₇: ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ", "color": "#483D8B", "symbol": "🌀", "min": 0.0, "max": 0.25},
    8: {"name": "τ₈: ΜΕΤΑ-ΣΤΑΘΕΡΟΤΗΤΑ", "color": "#000080", "symbol": "∞", "min": 0.95, "max": 1.5},
    9: {"name": "τ₉: ΜΕΤΑ-ΥΠΕΡΒΑΣΗ", "color": "#000000", "symbol": "🌟", "min": 1.5, "max": float('inf')}
}

# ============================================================================
# ΜΕΡΟΣ 2: XENOPOULOSUNITYFIELD - ΤΟ ΕΝΟΠΟΙΗΜΕΝΟ ΠΕΔΙΟ
# ============================================================================

class XenopoulosUnityField:
    """
    Το Ενοποιημένο Πεδίο του Ξενόπουλου (XenopoulosUnityField)

    Χ(x, t) = L(x) ⊕ ∫₀ᵗ S(τ) dτ

    όπου:
    - L(x): το στιγμιαίο layer (Αρχή 4: Υπέρβαση)
    - S(τ): το ιστορικό σύστημα (Αρχή 6: Αυτο-αναφορικότητα)
    - ⊕: τελεστής διαλεκτικής σύνθεσης (Αρχή 34: Ολοκλήρωση-Υπέρβαση)
    - Ανίχνευση παραδόξων (Αρχή 9: Παραδοξογένεση)
    """

    def __init__(self, w=0.5, r=0.15, horizon=100):
        self.w = w
        self.r = r
        self.horizon = horizon
        self.history = []          # Ιστορία τιμών Α
        self.field_history = []     # Ιστορία τιμών πεδίου
        self.time = 0

    # ------------------------------------------------------------------------
    # 1. ΣΤΙΓΜΙΑΙΟ ΜΕΡΟΣ (Layer) - Αρχή 4: Υπέρβαση
    # ------------------------------------------------------------------------
    def layer(self, x):
        """
        L(x) = tanh(x·w)·(1 - r)

        Το στιγμιαίο μέρος - η άμεση απόκριση στο ερέθισμα.
        """
        return np.tanh(x * self.w) * (1 - self.r)

    # ------------------------------------------------------------------------
    # 2. ΙΣΤΟΡΙΚΟ ΜΕΡΟΣ (System) - Αρχή 6: Αυτο-αναφορικότητα
    # ------------------------------------------------------------------------
    def system(self, t, A):
        """
        S(t) = ¬ᴰ(A) = -A·p·h·(1 + m) + ε

        Το ιστορικό μέρος - επηρεάζεται από το παρελθόν.
        """
        # Memory effect (αυτο-αναφορικότητα)
        if len(self.history) > 0:
            window = min(10, len(self.history))
            recent = np.mean(self.history[-window:])
            memory = 0.2 * np.tanh(recent * 2)
        else:
            memory = 0

        # Στοχαστικοί παράγοντες
        p = 0.7 + 0.3 * random.random()      # preservation
        h = 1.0 + 0.3 * random.random()      # historical weight
        noise = 0.05 * (1 + abs(A)) * random.gauss(0, 1)

        # Διαλεκτική άρνηση
        return -A * p * h * (1 + memory) + noise

    # ------------------------------------------------------------------------
    # 3. ΤΕΛΕΣΤΗΣ ΔΙΑΛΕΚΤΙΚΗΣ ΣΥΝΘΕΣΗΣ (⊕) - Αρχή 34: Ολοκλήρωση-Υπέρβαση
    # ------------------------------------------------------------------------
    def compose(self, a, b):
        """
        a ⊕ b = (a + b)·(1 - |tanh(a·b)|) + i·(a·b)

        Ο τελεστής που ενώνει στιγμιαίο και ιστορικό μέρος.
        Το αποτέλεσμα είναι ΜΙΓΑΔΙΚΟ:
        - Re: η σύνθεση (τι ισχυρίζεται το σύστημα)
        - Im: η ένταση (πόσο "ταράζεται" το σύστημα)
        """
        real_part = (a + b) * (1 - abs(np.tanh(a * b + 1e-10)))
        imag_part = a * b
        return real_part + 1j * imag_part

    # ------------------------------------------------------------------------
    # 4. ΥΠΟΛΟΓΙΣΜΟΣ ΕΝΤΑΣΗΣ
    # ------------------------------------------------------------------------
    def tension(self, X):
        """T = |Im(Χ)| - η διαλεκτική ένταση"""
        return abs(np.imag(X))

    # ------------------------------------------------------------------------
    # 5. ΤΑΞΙΝΟΜΗΣΗ ΣΤΑΔΙΟΥ - Αρχή 9: Παραδοξογένεση
    # ------------------------------------------------------------------------
    def classify_stage(self, X):
        """
        Ταξινόμηση σε στάδιο με βάση:
        - Re(Χ): τι ισχυρίζεται
        - Im(Χ): ένταση
        - Tension: κρυμμένη ένταση
        """
        Re = np.real(X)
        Im = np.imag(X)
        T = self.tension(X)

        # Στάδιο 6: ΠΑΡΑΔΟΞΟ (τ₆)
        if abs(Re) > 0.85 and abs(Im) > 0.85 and T < 0.4:
            return 6, STAGES[6]["name"]

        # Στάδιο 7: ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)
        if T < 0.25 and (abs(Re) > 0.75 or abs(Im) > 0.75):
            return 7, STAGES[7]["name"]

        # Κανονικά στάδια με βάση την ένταση
        for stage_id, stage_info in STAGES.items():
            if stage_id not in [6, 7]:  # Εξαίρεση ειδικών σταδίων
                if stage_info["min"] <= T < stage_info["max"]:
                    return stage_id, stage_info["name"]

        return 0, STAGES[0]["name"]

    # ------------------------------------------------------------------------
    # 6. ΥΠΟΛΟΓΙΣΜΟΣ XEPTQLRI
    # ------------------------------------------------------------------------
    def calculate_xeptqlri(self, X):
        """
        XEPTQLRI = [T^1.2 · trend · paradox · asym] / 0.85
        """
        Re = np.real(X)
        Im = np.imag(X)
        T = self.tension(X)

        # Base
        base = T ** 1.2

        # Trend factor (από ιστορία)
        if len(self.field_history) >= 5:
            recent = [np.imag(f) for f in self.field_history[-5:]]
            if len(recent) > 1:
                trend_coef = np.polyfit(range(len(recent)), recent, 1)[0]
                trend = 1.0 + abs(trend_coef) * 15
            else:
                trend = 1.0
        else:
            trend = 1.0

        # Paradox factor
        paradox = 1.0
        if abs(Re) > 0.85 and abs(Im) > 0.85:
            paradox = 2.8 if T < 0.4 else 2.0

        # Asymmetry factor
        asym = 1.0 + (1 - abs(abs(Re) - abs(Im))) * 0.5

        return min(5.0, max(0.0, (base * trend * paradox * asym) / 0.85))

    # ------------------------------------------------------------------------
    # 7. ΟΛΟΚΛΗΡΩΤΙΚΗ ΜΟΡΦΗ - ΤΟ ΠΛΗΡΕΣ ΠΕΔΙΟ
    # ------------------------------------------------------------------------
    def forward(self, x, A=None):
        """
        Χ(x, t) = L(x) ⊕ ∫₀ᵗ S(τ) dτ

        Υπολογίζει το πεδίο τη χρονική στιγμή t.
        """
        # Αν δεν δόθηκε Α, υπολόγισε από ιστορία
        if A is None:
            A = 0.5 + 0.3 * np.sin(self.time * 0.1) if len(self.history) > 0 else 0.5

        # Στιγμιαίο μέρος
        L = self.layer(x)

        # Ολοκλήρωμα ιστορικού μέρους
        integral = 0
        for tau in range(min(self.time, self.horizon)):
            if tau < len(self.history):
                integral += self.system(tau, self.history[tau])

        # Σύνθεση
        X = self.compose(L, integral)

        # Καταγραφή
        self.field_history.append(X)
        self.history.append(A)
        self.time += 1

        # Υπολογισμός μετρικών
        stage_id, stage_name = self.classify_stage(X)
        xeptqlri = self.calculate_xeptqlri(X)

        return {
            'field': X,
            'real': np.real(X),
            'imag': np.imag(X),
            'tension': self.tension(X),
            'stage_id': stage_id,
            'stage': stage_name,
            'xeptqlri': xeptqlri,
            'layer': L,
            'integral': integral,
            'time': self.time
        }

    # ------------------------------------------------------------------------
    # 8. ΕΝΤΟΠΙΣΜΟΣ ΠΑΡΑΔΟΞΩΝ ΣΕ ΚΕΙΜΕΝΟ
    # ------------------------------------------------------------------------
    def analyze_text(self, text):
        """
        Αναλύει ένα κείμενο για παράδοξα και αντιφάσεις.
        """
        # Χωρισμός σε προτάσεις
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        results = []

        for i, sentence in enumerate(sentences):
            # Υπολογισμός φορτίου πρότασης
            words = sentence.lower().split()

            # Ανίχνευση συναισθηματικού φορτίου
            positive_words = ['ευγνωμοσύνη', 'εμπιστοσύνη', 'χαρά', 'καλός', 'σωστός']
            negative_words = ['μηνύω', 'καταγγέλλω', 'ψέμα', 'λάθος', 'πρόβλημα']

            pos_count = sum(1 for w in words if w in positive_words)
            neg_count = sum(1 for w in words if w in negative_words)

            # Ανίχνευση χρονολογιών
            dates = re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', sentence)

            # Ανίχνευση ποσών
            amounts = re.findall(r'\d+[.,]?\d*\s*[€$]', sentence)

            # Υπολογισμός εισόδου για το πεδίο
            x = (pos_count - neg_count) / max(len(words), 1) + 0.5

            # Εφαρμογή πεδίου
            result = self.forward(x)
            results.append({
                'sentence': sentence[:50] + ('...' if len(sentence) > 50 else ''),
                'index': i,
                'field': result,
                'dates': dates,
                'amounts': amounts,
                'pos_neg': (pos_count, neg_count)
            })

        return results

    # ------------------------------------------------------------------------
    # 9. ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ
    # ------------------------------------------------------------------------
    def statistics(self):
        """Στατιστικά του πεδίου"""
        if not self.field_history:
            return {}

        real_parts = [np.real(f) for f in self.field_history]
        imag_parts = [np.imag(f) for f in self.field_history]

        return {
            'real_mean': float(np.mean(real_parts)),
            'real_std': float(np.std(real_parts)),
            'real_max': float(np.max(real_parts)),
            'real_min': float(np.min(real_parts)),
            'imag_mean': float(np.mean(imag_parts)),
            'imag_std': float(np.std(imag_parts)),
            'imag_max': float(np.max(imag_parts)),
            'imag_min': float(np.min(imag_parts)),
            'energy': float(np.sum([abs(f)**2 for f in self.field_history])),
            'steps': len(self.field_history)
        }

    # ------------------------------------------------------------------------
    # 10. RESET
    # ------------------------------------------------------------------------
    def reset(self):
        """Επαναφορά πεδίου"""
        self.history = []
        self.field_history = []
        self.time = 0


# ============================================================================
# ΜΕΡΟΣ 3: TEXT ANALYZER (Προσαρμοσμένος για UnityField)
# ============================================================================

class TextAnalyzer:
    """Ανάλυση κειμένου με UnityField"""

    def __init__(self):
        self.field = XenopoulosUnityField()
        self.analyses = []

    def lexical_contradictions(self, text):
        """Λεξικές αντιφάσεις (αλλά, όμως, κλπ)"""
        words = ['αλλά', 'όμως', 'παρόλα', 'αντίθετα', 'μολονότι', 'ωστόσο', 'ενώ']
        count = 0
        sentences = [s for s in text.split('.') if s.strip()]

        for i, s in enumerate(sentences):
            found = [w for w in words if w in s.lower()]
            if found:
                count += 1

        return count

    def semantic_analysis(self, text):
        """Νοηματική ανάλυση με UnityField"""
        self.field.reset()
        results = self.field.analyze_text(text)

        # Συγκεντρωτικά στατιστικά
        real_vals = [r['field']['real'] for r in results]
        imag_vals = [r['field']['imag'] for r in results]
        tensions = [r['field']['tension'] for r in results]
        stages = [r['field']['stage_id'] for r in results]

        # Ανίχνευση παραδόξων
        paradoxes = sum(1 for s in stages if s == 6)
        false_stability = sum(1 for s in stages if s == 7)

        return {
            'words': len(text.split()),
            'sentences': len(results),
            'unique_words': len(set(text.lower().split())),
            'lexical_diversity': len(set(text.lower().split())) / max(len(text.split()), 1),
            'avg_sentence_length': len(text.split()) / max(len(results), 1),
            'readability': max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)),
            'lexical_contradictions': self.lexical_contradictions(text),
            'semantic_contradictions': paradoxes + false_stability,
            'paradoxes': paradoxes,
            'false_stability': false_stability,
            'avg_real': float(np.mean(real_vals)) if real_vals else 0,
            'avg_imag': float(np.mean(imag_vals)) if imag_vals else 0,
            'avg_tension': float(np.mean(tensions)) if tensions else 0,
            'max_tension': float(np.max(tensions)) if tensions else 0,
            'score': float(0.3 * (len(set(text.lower().split()))/max(len(text.split()),1)) +
                          0.4 * max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)) +
                          0.3 * (1 - (paradoxes + false_stability)/max(len(results),1))),
            'quality': self._get_quality(0.3 * (len(set(text.lower().split()))/max(len(text.split()),1)) +
                                         0.4 * max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)) +
                                         0.3 * (1 - (paradoxes + false_stability)/max(len(results),1)))
        }

    def _get_quality(self, score):
        if score > 0.8: return "ΕΞΑΙΡΕΤΙΚΗ"
        if score > 0.6: return "ΚΑΛΗ"
        if score > 0.4: return "ΜΕΤΡΙΑ"
        return "ΧΑΜΗΛΗ"

    def full_metrics(self, text):
        """Πλήρεις μετρικές"""
        semantic = self.semantic_analysis(text)

        # Υπολογισμός XEPTQLRI από τελευταία πρόταση
        self.field.reset()
        results = self.field.analyze_text(text)
        last_xeptqlri = results[-1]['field']['xeptqlri'] if results else 0

        return {
            'Βασικά': {
                'Χαρακτήρες': len(text),
                'Λέξεις': semantic['words'],
                'Προτάσεις': semantic['sentences']
            },
            'Γλωσσικά': {
                'Λεξιλογική ποικιλία': f"{semantic['lexical_diversity']*100:.1f}%",
                'Μ.Ο. μήκος πρότασης': f"{semantic['avg_sentence_length']:.1f}",
                'Αναγνωσιμότητα': f"{semantic['readability']*100:.1f}%"
            },
            'Αντιφάσεις': {
                'Λεξικές/Συντακτικές': semantic['lexical_contradictions'],
                'Νοηματικές': semantic['semantic_contradictions'],
                'Παράδοξα (τ₆)': semantic['paradoxes'],
                'Ψευδής σταθερότητα (τ₇)': semantic['false_stability'],
                'Σύνολο': semantic['lexical_contradictions'] + semantic['semantic_contradictions']
            },
            'XEPTQLRI': {
                'Δείκτης': f"{last_xeptqlri:.4f}",
                'Μέση ένταση': f"{semantic['avg_tension']:.4f}",
                'Μέγιστη ένταση': f"{semantic['max_tension']:.4f}"
            },
            'Ποιότητα': semantic['quality']
        }

    def final_assessment(self, text):
        """Τελική αποτίμηση με UnityField"""
        semantic = self.semantic_analysis(text)

        result = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║              ΤΕΛΙΚΗ ΑΠΟΤΙΜΗΣΗ ΜΕ XENOPOULOSUNITYFIELD                    ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  ⚠️  ΠΙΝΑΚΑΣ ΑΝΤΙΦΑΣΕΩΝ                                                  ║
║  ────────────────────────────────────────────────────────────────────── ║
║                                                                          ║
║  Τύπος Αντίφασης                  |  Πλήθος  |  Σοβαρότητα              ║
║  ────────────────────────────────────────────────────────────────────── ║
║  Λεξικές/Συντακτικές                |  {semantic['lexical_contradictions']:4d}      |  25%                       ║
║  Νοηματικές (σύνολο)                |  {semantic['semantic_contradictions']:4d}      |  Ποικίλη                   ║
║    • Παράδοξα (τ₆)                   |  {semantic['paradoxes']:4d}      |  85% (ΚΡΙΣΙΜΟ)              ║
║    • Ψευδής Σταθερότητα (τ₇)         |  {semantic['false_stability']:4d}      |  70% (ΥΠΟΠΤΟ)               ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  🔴 ΑΝΑΛΥΣΗ ΠΕΔΙΟΥ:                                                      ║
║     • Μέσο Re(Χ): {semantic['avg_real']:.4f} (τι ισχυρίζεται)                           ║
║     • Μέσο Im(Χ): {semantic['avg_imag']:.4f} (διαλεκτική ένταση)                        ║
║     • Μέση ένταση: {semantic['avg_tension']:.4f}                                        ║
║     • Μέγιστη ένταση: {semantic['max_tension']:.4f}                                      ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  🔍 ΕΝΤΟΠΙΣΜΕΝΑ ΠΑΡΑΔΟΞΑ:                                                ║
"""

        if semantic['paradoxes'] > 0:
            result += f"""
║     • Βρέθηκαν {semantic['paradoxes']} ΠΑΡΑΔΟΞΑ (τ₆) - ΑΜΕΣΗ ΠΡΟΣΟΧΗ!    ║
║     • Το κείμενο περιέχει ΛΟΓΙΚΕΣ ΑΝΤΙΦΑΣΕΙΣ                             ║
"""
        else:
            result += """
║     • Δεν βρέθηκαν παράδοξα (τ₆) - ΚΑΛΟ!                                 ║
"""

        if semantic['false_stability'] > 0:
            result += f"""
║     • Βρέθηκαν {semantic['false_stability']} περιπτώσεις ΨΕΥΔΟΥΣ ΣΤΑΘΕΡΟΤΗΤΑΣ (τ₇)   ║
║     • Υπάρχουν ΚΡΥΦΕΣ αντιφάσεις - ΠΡΟΣΟΧΗ!                              ║
"""

        result += f"""
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  💡 ΣΧΕΔΙΟ ΔΡΑΣΗΣ:                                                       ║
║                                                                          ║
║  ΦΑΣΗ 1 - Άμεσες ενέργειες (Προτεραιότητα ★★★):                         ║
"""

        if semantic['paradoxes'] > 0:
            result += f"""
║     • Διόρθωσε {semantic['paradoxes']} ΠΑΡΑΔΟΞΑ (λογικές αντιφάσεις)     ║
"""
        else:
            result += """
║     • Καμία άμεση ενέργεια για παράδοξα - ΠΡΟΧΩΡΑ                         ║
"""

        result += f"""
║                                                                          ║
║  ΦΑΣΗ 2 - Βελτιστοποίηση (Προτεραιότητα ✦):                             ║
║     • Διόρθωσε {semantic['lexical_contradictions']} λεξικές/συντακτικές αντιφάσεις   ║
║     • Διόρθωσε {semantic['false_stability']} περιπτώσεις ψευδούς σταθερότητας       ║
║                                                                          ║
║  ΦΑΣΗ 3 - Τελικός έλεγχος:                                               ║
║     • Επαναξιολόγηση με UnityField                                      ║
║     • Σύγκριση αρχικού-διορθωμένου                                       ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
        return result

    def unity_field_report(self, text):
        """Αναλυτική αναφορά UnityField"""
        self.field.reset()
        results = self.field.analyze_text(text)

        report = f"""
🌀 XENOPOULOS UNITY FIELD - ΑΝΑΛΥΤΙΚΗ ΑΝΑΦΟΡΑ
================================================================================

📊 ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ:
   • Βήματα: {len(results)}
   • Τελικό Re(Χ): {results[-1]['field']['real']:.4f}
   • Τελικό Im(Χ): {results[-1]['field']['imag']:.4f}
   • Τελική ένταση: {results[-1]['field']['tension']:.4f}
   • Τελικό στάδιο: {results[-1]['field']['stage']}
   • Τελικό XEPTQLRI: {results[-1]['field']['xeptqlri']:.4f}

🔍 ΑΝΑΛΥΣΗ ΑΝΑ ΠΡΟΤΑΣΗ:
"""
        for i, r in enumerate(results[:10]):  # Πρώτες 10 προτάσεις
            report += f"""
   {i+1}. {r['sentence']}
       • Re={r['field']['real']:.3f}, Im={r['field']['imag']:.3f}, T={r['field']['tension']:.3f}
       • Στάδιο: {r['field']['stage']}
       • Ημερομηνίες: {r['dates']}, Ποσά: {r['amounts']}
"""

        if len(results) > 10:
            report += f"\n   ... και {len(results)-10} ακόμη προτάσεις"

        stats = self.field.statistics()
        report += f"""

📈 ΣΥΝΟΛΙΚΑ ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ:
   • Μέσο Re(Χ): {stats.get('real_mean', 0):.4f}
   • Μέσο Im(Χ): {stats.get('imag_mean', 0):.4f}
   • Μέγιστη ένταση: {stats.get('imag_max', 0):.4f}
   • Ολική ενέργεια: {stats.get('energy', 0):.4f}
   • Βήματα: {stats.get('steps', 0)}

🏆 ΤΕΛΙΚΗ ΔΙΑΓΝΩΣΗ:
"""
        last = results[-1]['field']
        if last['stage_id'] == 6:
            report += "🔴 ΤΟ ΚΕΙΜΕΝΟ ΠΕΡΙΕΧΕΙ ΠΑΡΑΔΟΞΑ - ΑΠΑΙΤΕΙΤΑΙ ΔΙΟΡΘΩΣΗ!"
        elif last['stage_id'] == 7:
            report += "⚠️ ΤΟ ΚΕΙΜΕΝΟ ΕΧΕΙ ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ - ΥΠΑΡΧΟΥΝ ΚΡΥΦΕΣ ΑΝΤΙΦΑΣΕΙΣ"
        elif last['tension'] < 0.3:
            report += "✅ ΤΟ ΚΕΙΜΕΝΟ ΕΙΝΑΙ ΛΟΓΙΚΑ ΣΥΝΕΠΕΣ"
        else:
            report += f"📊 ΤΟ ΚΕΙΜΕΝΟ ΒΡΙΣΚΕΤΑΙ ΣΕ ΚΑΤΑΣΤΑΣΗ: {last['stage']}"

        return report


# ============================================================================
# ΜΕΡΟΣ 4: ΑΣΦΑΛΕΙΑ
# ============================================================================

class SecuritySystem:
    def __init__(self):
        self.level = 'user'
        self.keys = ['15081920', '15-08-1920', '15 Αυγούστου 1920']

    def authenticate(self, key):
        if key in self.keys:
            self.level = 'admin'
            return True
        return False


# ============================================================================
# ΜΕΡΟΣ 5: MEMORY MANAGER
# ============================================================================

class MemoryManager:
    def __init__(self, memory_file="exdt_memory.json"):
        self.memory_file = memory_file

    def save(self, data):
        with open(self.memory_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False, default=str)
        return True

    def load(self):
        if os.path.exists(self.memory_file):
            with open(self.memory_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        return None


# ============================================================================
# ΜΕΡΟΣ 6: ΠΛΗΡΕΣ ΣΥΣΤΗΜΑ
# ============================================================================

class XenopoulosSystem:
    def __init__(self, memory_file="exdt_memory.json"):
        self.text = TextAnalyzer()
        self.unity_field = XenopoulosUnityField()
        self.security = SecuritySystem()
        self.memory = MemoryManager(memory_file)
        self.analyses = []
        self.load_memory()

    def load_memory(self):
        data = self.memory.load()
        if data:
            self.analyses = data.get('analyses', [])
            print(f"📂 Φορτώθηκαν {len(self.analyses)} προηγούμενες αναλύσεις")

    def save_memory(self):
        data = {'analyses': self.analyses, 'last_save': datetime.now().isoformat()}
        self.memory.save(data)

    def analyze_text(self, text, save=True):
        result = {
            'id': str(uuid.uuid4())[:8],
            'timestamp': datetime.now().isoformat(),
            'text_preview': text[:100] + ('...' if len(text) > 100 else ''),
            'semantic': self.text.semantic_analysis(text),
            'unity': self.text.unity_field_report(text)
        }

        if save:
            self.analyses.append(result)
            if len(self.analyses) % 5 == 0:
                self.save_memory()

        return result

    def info(self):
        return {
            'version': VERSION,
            'access': self.security.level,
            'analyses': len(self.analyses)
        }


# ============================================================================
# ΜΕΡΟΣ 7: UI
# ============================================================================

class XenopoulosUI:
    def __init__(self, system):
        self.sys = system
        self.text = system.text
        if IPYWIDGETS_AVAILABLE:
            self._build()

    def _build(self):
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #667eea, #764ba2);
             padding: 20px; border-radius: 15px; color: white; text-align: center;">
            <h1>🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ</h1>
            <h2>EXDT v3.0 με XENOPOULOSUNITYFIELD</h2>
            <p>9 Λειτουργίες Ανάλυσης | 34 Αρχές | 7 Θεωρήματα</p>
        </div>
        """))

        self.input = widgets.Textarea(
            value='Επικολλήστε το κείμενό σας εδώ...',
            layout=widgets.Layout(width='100%', height='150px')
        )
        display(self.input)

        # 9 κουμπιά
        btns = []
        btn_configs = [
            ('1️⃣ ΛΕΞΙΚΕΣ', 'primary', self._lexical),
            ('2️⃣ ΝΟΗΜΑΤΙΚΗ', 'success', self._semantic),
            ('3️⃣ ΔΙΟΡΘΩΣΗ', 'info', self._correct),
            ('4️⃣ ΛΙΣΤΑ', 'warning', self._list),
            ('5️⃣ ΜΕΤΡΙΚΕΣ', 'danger', self._metrics),
            ('6️⃣ ΣΥΝΟΛΙΚΗ', 'primary', self._report),
            ('7️⃣ ΤΕΛΙΚΗ v3', 'success', self._final),
            ('8️⃣ EXTREME', 'danger', self._extreme),
            ('0️⃣ UNITY FIELD', 'info', self._unity)
        ]

        for txt, style, handler in btn_configs:
            btn = widgets.Button(description=txt, button_style=style,
                               layout=widgets.Layout(width='160px', margin='2px'))
            btn.on_click(handler)
            btns.append(btn)

        display(widgets.HBox(btns[:5]))
        display(widgets.HBox(btns[5:]))

        # Output
        self.out = widgets.Output(layout=widgets.Layout(width='100%', height='500px', overflow='auto'))
        display(self.out)

    def _get_text(self):
        t = self.input.value
        if t in ['Επικολλήστε το κείμενό σας εδώ...', '']:
            with self.out:
                clear_output()
                print("❌ Παρακαλώ εισάγετε κείμενο")
            return None
        return t

    def _lexical(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            c = self.text.lexical_contradictions(t)
            print(f"🔍 ΛΕΞΙΚΕΣ ΑΝΤΙΦΑΣΕΙΣ: {c}")

    def _semantic(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            m = self.text.semantic_analysis(t)
            print("📊 ΝΟΗΜΑΤΙΚΗ ΑΝΑΛΥΣΗ\n" + "="*50)
            for k, v in m.items():
                if k not in ['avg_real', 'avg_imag', 'avg_tension', 'max_tension', 'paradoxes', 'false_stability']:
                    print(f"{k}: {v}")
            print(f"\n📈 ΣΤΑΤΙΣΤΙΚΑ UNITY FIELD:")
            print(f"   Μέσο Re(Χ): {m['avg_real']:.4f}")
            print(f"   Μέσο Im(Χ): {m['avg_imag']:.4f}")
            print(f"   Μέση ένταση: {m['avg_tension']:.4f}")
            print(f"   Μέγιστη ένταση: {m['max_tension']:.4f}")
            print(f"   Παράδοξα (τ₆): {m['paradoxes']}")
            print(f"   Ψευδής σταθερότητα (τ₇): {m['false_stability']}")

    def _correct(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print("✏️ ΔΙΟΡΘΩΣΗ\n" + "="*50)
            print("Η λειτουργία διόρθωσης θα ενσωματωθεί σύντομα")

    def _list(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            self.text.field.reset()
            results = self.text.field.analyze_text(t)
            print("📋 ΛΕΠΤΟΜΕΡΗΣ ΛΙΣΤΑ ΑΝΑΛΥΣΗΣ\n" + "="*50)
            for i, r in enumerate(results[:10]):
                print(f"\n{i+1}. {r['sentence']}")
                print(f"   Re={r['field']['real']:.3f}, Im={r['field']['imag']:.3f}")
                print(f"   Στάδιο: {r['field']['stage']}")
                if r['dates']:
                    print(f"   Ημερομηνίες: {r['dates']}")
                if r['amounts']:
                    print(f"   Ποσά: {r['amounts']}")

    def _metrics(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            m = self.text.full_metrics(t)
            print("📊 ΠΛΗΡΕΙΣ ΜΕΤΡΙΚΕΣ\n" + "="*50)
            for category, values in m.items():
                print(f"\n{category}:")
                if isinstance(values, dict):
                    for k, v in values.items():
                        print(f"  {k}: {v}")
                else:
                    print(f"  {values}")

    def _report(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.final_assessment(t))

    def _final(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.final_assessment(t))

    def _extreme(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            self.text.field.reset()
            results = self.text.field.analyze_text(t)
            last = results[-1]['field']
            print("🔥 ΑΝΑΛΥΣΗ EXTREME ΚΑΤΑΣΤΑΣΗΣ\n" + "="*50)
            print(f"XEPTQLRI: {last['xeptqlri']:.4f}")
            if last['xeptqlri'] >= 2.0:
                print("🚨 EXTREME ΚΑΤΑΣΤΑΣΗ! (XEPTQLRI > 2.0)")
            elif last['xeptqlri'] >= 1.5:
                print("⚠️ ΥΠΕΡΚΡΙΣΙΜΗ ΚΑΤΑΣΤΑΣΗ")
            elif last['stage_id'] == 6:
                print("🔴 ΠΑΡΑΔΟΞΟ (τ₆)")
            elif last['stage_id'] == 7:
                print("⚠️ ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)")
            else:
                print("✅ ΦΥΣΙΟΛΟΓΙΚΗ ΚΑΤΑΣΤΑΣΗ")

    def _unity(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.unity_field_report(t))


# ============================================================================
# ΜΕΡΟΣ 8: ΕΚΚΙΝΗΣΗ
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ - ΔΙΑΛΕΚΤΙΚΟΣ ΜΕΤΑΣΧΗΜΑΤΙΣΤΗΣ")
    print("="*80)
    print("📌 34 Αρχές | 7 Θεωρήματα | 9 Λειτουργίες | XENOPOULOSUNITYFIELD")
    print("📅 Ημερομηνία γέννησης: 15 Αυγούστου 1920")
    print("🔑 Κλειδί: 15081920 | 15-08-1920 | 15 Αυγούστου 1920")
    print("="*80)

    system = XenopoulosSystem()

    if IPYWIDGETS_AVAILABLE:
        ui = XenopoulosUI(system)
        print("\n✅ Σύστημα έτοιμο - 9 λειτουργίες (ΝΕΟ: 0️⃣ UNITY FIELD)")
    else:
        print("⚠️ Εκτέλεση σε console mode")

    print("\n" + "="*80)
    print("🚀 ΝΕΑ ΛΕΙΤΟΥΡΓΙΑ: XENOPOULOSUNITYFIELD")
    print("   • Ενοποιεί Layer (στιγμιαίο) και System (ιστορικό)")
    print("   • Ανιχνεύει ΠΑΡΑΔΟΞΑ (τ₆) και ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)")
    print("   • Χρησιμοποιεί μιγαδική ανάλυση (Re + i·Im)")
    print("="*80)

    # ============================================================================
# ΑΥΤΟΜΑΤΗ ΑΠΟΘΗΚΕΥΣΗ ΟΛΩΝ ΤΩΝ ΑΠΟΤΕΛΕΣΜΑΤΩΝ ΚΑΙ ΓΡΑΦΗΜΑΤΩΝ
# ============================================================================

import json
import pickle
from datetime import datetime
import os
import matplotlib.pyplot as plt
import numpy as np

class EXDTAutoSave:
    """
    Αυτόματη αποθήκευση ΟΛΩΝ των αποτελεσμάτων και γραφημάτων
    """

    def __init__(self, session_name=None):
        # Δημιουργία μοναδικού ονόματος session
        if session_name is None:
            self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        else:
            self.session_id = session_name

        # Δημιουργία φακέλων
        self.base_dir = f"EXDT_Session_{self.session_id}"
        self.text_dir = f"{self.base_dir}/text_reports"
        self.graphs_dir = f"{self.base_dir}/graphs"
        self.json_dir = f"{self.base_dir}/json_data"
        self.summary_dir = f"{self.base_dir}/summary"

        for dir_path in [self.text_dir, self.graphs_dir, self.json_dir, self.summary_dir]:
            os.makedirs(dir_path, exist_ok=True)

        # Αρχείο καταγραφής session
        self.log_file = f"{self.base_dir}/session_log.txt"
        self._log(f"🔥 NEW EXDT SESSION: {self.session_id}")
        self._log(f"📁 Results saved in: {self.base_dir}")

        # Συλλογή όλων των αποτελεσμάτων
        self.all_unity_reports = []
        self.all_metrics = []
        self.all_extreme = []
        self.graph_count = 0

    def _log(self, message):
        """Εσωτερική καταγραφή"""
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(f"{datetime.now().strftime('%H:%M:%S')} - {message}\n")
        print(message)

    # ------------------------------------------------------------------------
    # 1. ΑΠΟΘΗΚΕΥΣΗ ΓΡΑΦΗΜΑΤΩΝ
    # ------------------------------------------------------------------------

    def save_current_figure(self, name, fig=None, dpi=300, formats=['png', 'pdf', 'svg']):
        """
        Αποθήκευση του τρέχοντος γραφήματος σε πολλαπλές μορφές
        """
        if fig is None:
            fig = plt.gcf()

        timestamp = datetime.now().strftime("%H%M%S")
        self.graph_count += 1

        saved_files = []
        for fmt in formats:
            filename = f"{self.graphs_dir}/{self.graph_count:03d}_{name}_{timestamp}.{fmt}"
            fig.savefig(filename, dpi=dpi, bbox_inches='tight', format=fmt)
            saved_files.append(filename)

        self._log(f"📊 Γράφημα {self.graph_count}: {name} αποθηκεύτηκε σε {len(saved_files)} μορφές")
        return saved_files

    def save_unity_field_plots(self, results, name="unity_field"):
        """
        Αποθήκευση όλων των γραφημάτων του Unity Field
        """
        if not results:
            return

        # Δημιουργία figure με 4 υπογραφήματα
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'XenopoulosUnityField - {name}', fontsize=16)

        steps = range(len(results))
        real_parts = [r['field']['real'] for r in results]
        imag_parts = [r['field']['imag'] for r in results]
        tensions = [r['field']['tension'] for r in results]

        # 1. Re(Χ) και Im(Χ)
        ax1 = axes[0, 0]
        ax1.plot(steps, real_parts, 'b-', label='Re(Χ)', linewidth=2)
        ax1.plot(steps, imag_parts, 'r--', label='Im(Χ)', linewidth=2)
        ax1.axhline(y=0.85, color='purple', linestyle=':', label='Όριο παραδόξου')
        ax1.set_xlabel('Χρονικό Βήμα')
        ax1.set_ylabel('Τιμή')
        ax1.set_title('Πραγματικό και Φανταστικό Μέρος')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2. Διαλεκτική Ένταση
        ax2 = axes[0, 1]
        ax2.plot(steps, tensions, 'purple', linewidth=2)
        ax2.axhline(y=0.4, color='r', linestyle='--', label='Όριο παραδόξου')
        ax2.axhline(y=0.25, color='orange', linestyle='--', label='Όριο ψευδούς σταθ.')
        ax2.fill_between(steps, 0, tensions, alpha=0.3, color='purple')
        ax2.set_xlabel('Χρονικό Βήμα')
        ax2.set_ylabel('Ένταση |Im(Χ)|')
        ax2.set_title('Διαλεκτική Ένταση')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3. Φασικό διάγραμμα
        ax3 = axes[1, 0]
        scatter = ax3.scatter(real_parts, imag_parts, c=steps, cmap='viridis',
                            alpha=0.7, s=50)
        plt.colorbar(scatter, ax=ax3, label='Χρονικό Βήμα')
        ax3.axhline(y=0.85, color='r', linestyle='--', alpha=0.5)
        ax3.axvline(x=0.85, color='r', linestyle='--', alpha=0.5)
        ax3.set_xlabel('Re(Χ)')
        ax3.set_ylabel('Im(Χ)')
        ax3.set_title('Φασικό Διάγραμμα')
        ax3.grid(True, alpha=0.3)

        # 4. Ιστόγραμμα
        ax4 = axes[1, 1]
        ax4.hist(tensions, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
        ax4.axvline(x=0.4, color='r', linestyle='--', label='Όριο παραδόξου')
        ax4.axvline(x=0.25, color='orange', linestyle='--', label='Όριο ψευδούς')
        ax4.set_xlabel('Διαλεκτική Ένταση')
        ax4.set_ylabel('Συχνότητα')
        ax4.set_title('Κατανομή Εντάσεων')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()

        # Αποθήκευση
        return self.save_current_figure(f"unity_field_{name}", fig)

    # ------------------------------------------------------------------------
    # 2. ΑΠΟΘΗΚΕΥΣΗ ΑΝΑΦΟΡΩΝ
    # ------------------------------------------------------------------------

    def save_unity_report(self, report_text, name="unity_report"):
        """Αποθήκευση αναφοράς Unity Field"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.text_dir}/{name}_{timestamp}.txt"

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_text)

        self._log(f"📝 Αναφορά αποθηκεύτηκε: {filename}")

        # Επίσης αποθήκευση σε JSON
        self.save_json({
            'type': 'unity_report',
            'timestamp': datetime.now().isoformat(),
            'report': report_text,
            'report_length': len(report_text)
        }, f"unity_report_{timestamp}")

        self.all_unity_reports.append({
            'filename': filename,
            'timestamp': datetime.now().isoformat()
        })

        return filename

    def save_metrics(self, metrics_dict, name="metrics"):
        """Αποθήκευση μετρικών"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.json_dir}/{name}_{timestamp}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(metrics_dict, f, indent=2, ensure_ascii=False, default=str)

        self._log(f"📊 Μετρικές αποθηκεύτηκαν: {filename}")
        self.all_metrics.append(metrics_dict)

        # Δημιουργία και σύνοψης
        self._create_metrics_summary(metrics_dict, timestamp)

        return filename

    def save_extreme_analysis(self, result, name="extreme"):
        """Αποθήκευση extreme analysis"""

        timestamp = datetime.now().strftime("%H%M%S")

        # Κείμενο
        text_file = f"{self.text_dir}/{name}_{timestamp}.txt"
        with open(text_file, 'w', encoding='utf-8') as f:
            if 'xeptqlri' in result:
                f.write(f"XEPTQLRI: {result['xeptqlri']}\n")
                if result['xeptqlri'] >= 2.0:
                    f.write("🚨 EXTREME ΚΑΤΑΣΤΑΣΗ!\n")

        # JSON
        json_file = f"{self.json_dir}/{name}_{timestamp}.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, default=str)

        self._log(f"🔥 Extreme analysis: {text_file}")
        self.all_extreme.append(result)

        return text_file

    def save_json(self, data, name="data"):
        """Γενική αποθήκευση JSON"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.json_dir}/{name}_{timestamp}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False, default=str)

        return filename

    # ------------------------------------------------------------------------
    # 3. ΣΥΝΟΨΕΙΣ ΚΑΙ ΣΤΑΤΙΣΤΙΚΑ
    # ------------------------------------------------------------------------

    def _create_metrics_summary(self, metrics, timestamp):
        """Δημιουργία σύνοψης από μετρικές"""

        summary_file = f"{self.summary_dir}/summary_{timestamp}.txt"

        with open(summary_file, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write("📊 ΣΥΝΟΨΗ ΜΕΤΡΙΚΩΝ\n")
            f.write("="*60 + "\n\n")

            if 'Βασικά' in metrics:
                f.write("📌 ΒΑΣΙΚΑ ΣΤΟΙΧΕΙΑ:\n")
                for k, v in metrics['Βασικά'].items():
                    f.write(f"   {k}: {v}\n")
                f.write("\n")

            if 'XEPTQLRI' in metrics:
                f.write("🔥 XEPTQLRI:\n")
                for k, v in metrics['XEPTQLRI'].items():
                    f.write(f"   {k}: {v}\n")
                f.write("\n")

            if 'Ποιότητα' in metrics:
                f.write(f"⭐ ΠΟΙΟΤΗΤΑ: {metrics['Ποιότητα']}\n")

        self._log(f"📋 Σύνοψη δημιουργήθηκε: {summary_file}")

    def create_final_report(self):
        """Δημιουργία τελικής αναφοράς όλης της συνεδρίας"""

        report_file = f"{self.base_dir}/FINAL_REPORT.txt"

        with open(report_file, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("🏛️ EXDT v3.0 - ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ ΣΥΝΕΔΡΙΑΣ\n")
            f.write(f"📅 Session ID: {self.session_id}\n")
            f.write(f"🕒 Ημερομηνία: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")

            f.write("📊 ΣΤΑΤΙΣΤΙΚΑ ΣΥΝΕΔΡΙΑΣ:\n")
            f.write(f"   • Unity Reports: {len(self.all_unity_reports)}\n")
            f.write(f"   • Μετρικές: {len(self.all_metrics)}\n")
            f.write(f"   • Extreme Analyses: {len(self.all_extreme)}\n")
            f.write(f"   • Γραφήματα: {self.graph_count}\n\n")

            if self.all_metrics:
                f.write("🔥 ΜΕΓΙΣΤΕΣ ΕΝΤΑΣΕΙΣ:\n")
                for i, m in enumerate(self.all_metrics[-5:]):  # Τελευταίες 5
                    if 'XEPTQLRI' in m:
                        intensity = m['XEPTQLRI'].get('Μέγιστη ένταση', 'N/A')
                        f.write(f"   • Ανάλυση {i+1}: {intensity}\n")

            f.write("\n" + "="*80 + "\n")
            f.write("📁 ΟΛΑ ΤΑ ΑΡΧΕΙΑ ΑΠΟΘΗΚΕΥΤΗΚΑΝ\n")
            f.write(f"📂 Φάκελος: {self.base_dir}\n")

        self._log(f"\n🎉 ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ: {report_file}")
        self._log(f"📁 ΟΛΑ ΤΑ ΑΡΧΕΙΑ ΣΤΟΝ ΦΑΚΕΛΟ: {self.base_dir}")

        return report_file

    # ------------------------------------------------------------------------
    # 4. ΒΟΗΘΗΤΙΚΕΣ ΣΥΝΑΡΤΗΣΕΙΣ
    # ------------------------------------------------------------------------

    def zip_session(self):
        """Δημιουργία zip αρχείου με όλη τη συνεδρία"""

        import shutil

        zip_filename = f"{self.base_dir}.zip"
        shutil.make_archive(self.base_dir, 'zip', self.base_dir)

        self._log(f"📦 Συνεδρία συμπιέστηκε: {zip_filename}")
        return zip_filename

    def download_session(self):
        """Αυτόματο download του zip στο Colab"""

        try:
            from google.colab import files
            zip_file = self.zip_session()
            files.download(zip_file)
            self._log("📥 Το αρχείο κατέβηκε στον υπολογιστή σου!")
        except:
            self._log("⚠️ Δεν μπορεί να γίνει αυτόματο download (μη-Colab περιβάλλον)")


# ============================================================================
# ΠΑΡΑΔΕΙΓΜΑ ΧΡΗΣΗΣ
# ============================================================================

# Δημιουργία αυτόματης αποθήκευσης
saver = EXDTAutoSave("DeepSeek_Analysis_19_3_2026")

# ΜΕΤΑ ΑΠΟ ΚΑΘΕ ΑΝΑΛΥΣΗ:

# 1. Για αποθήκευση αναφοράς Unity Field:
# saver.save_unity_report(unity_report_text, "deepseek_analysis_1")

# 2. Για αποθήκευση γραφημάτων:
# saver.save_unity_field_plots(results, "deepseek_analysis")

# 3. Για αποθήκευση μετρικών:
# saver.save_metrics(metrics_dict, "deepseek_metrics")

# 4. ΣΤΟ ΤΕΛΟΣ:
# saver.create_final_report()
# saver.download_session()  # Κατεβάζει όλα τα αρχεία!# ============================================================================
# EPAMEINONDAS XENOPOULOS DIALECTICAL TRANSFORMER (EXDT) v3.0
# ΜΕ XENOPOULOSUNITYFIELD - ΠΛΗΡΗΣ ΣΥΓΧΩΝΕΥΣΗ LAYER & SYSTEM
# ============================================================================

# ----------------------------------------------------------------------------
# ΜΕΡΟΣ 0: IMPORTS
# ----------------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import uuid
import os
import sys
import json
import hashlib
import re
import math
import random
from collections import defaultdict
import getpass

# Colab/Jupyter
try:
    from IPython.display import display, HTML, clear_output
    import ipywidgets as widgets
    IN_COLAB = True
    IPYWIDGETS_AVAILABLE = True
except ImportError:
    IN_COLAB = False
    IPYWIDGETS_AVAILABLE = False
    display = print
    HTML = lambda x: x
    clear_output = lambda: None

# ============================================================================
# ΜΕΡΟΣ 1: ΣΤΑΘΕΡΕΣ ΚΑΙ ΠΑΡΑΜΕΤΡΟΙ
# ============================================================================

VERSION = "EXDT-3.0-UNITY"
CREATOR = "Epameinondas Xenopoulos"
BIRTH_DATE = "1920-08-15"

# Χρονικά κατώφλια
TIME_THRESHOLDS = {i: max(0.45, 0.75 - i*0.01) for i in range(1, 35)}
TIME_THRESHOLDS.update({i: 0.75 for i in range(1, 6)})

# Παράμετροι
P = {
    'α1': 0.35, 'α2': 0.28, 'α3': 0.42, 'α4': 0.19,
    'α5': 0.75, 'α6': 0.63, 'α7': 0.81, 'α8': 0.57,
    'α9': 0.44, 'α10': 0.92, 'α11': 0.38, 'α12': 0.73,
    'β1': 0.25, 'γ1': 0.15, 'δ1': 0.30, 'ε1': 0.20,
    'ζ1': 0.40, 'η1': 0.18, 'θ1': 0.22, 'ι1': 0.28,
    'κ1': 0.32, 'λ1': 0.38, 'μ1': 0.42, 'ν1': 0.48,
    'ξ1': 0.52, 'ο1': 0.58, 'π1': 0.62, 'ρ1': 0.68,
    'σ1': 0.72, 'τ1': 0.78, 'υ1': 0.82, 'φ1': 0.88,
    'χ1': 0.92, 'ψ1': 0.96, 'ω1': 0.99
}

# Στάδια διαλεκτικής εξέλιξης
STAGES = {
    0: {"name": "τ₀: ΣΥΝΟΧΗ", "color": "#2E8B57", "symbol": "✅", "min": 0.0, "max": 0.15},
    1: {"name": "τ₁: ΠΡΩΤΗ ΑΝΩΜΑΛΙΑ", "color": "#3CB371", "symbol": "⚠️", "min": 0.15, "max": 0.30},
    2: {"name": "τ₂: ΕΠΑΝΑΛΗΨΗ ΑΝΩΜΑΛΙΑΣ", "color": "#FFD700", "symbol": "🔄", "min": 0.30, "max": 0.45},
    3: {"name": "τ₃: ΣΥΣΣΩΡΕΥΣΗ ΕΝΤΑΣΗΣ", "color": "#FFA500", "symbol": "📈", "min": 0.45, "max": 0.60},
    4: {"name": "τ₄: ΣΗΜΕΙΟ ΚΡΙΣΗΣ", "color": "#FF6346", "symbol": "⚡", "min": 0.60, "max": 0.75},
    5: {"name": "τ₅: ΠΟΙΟΤΙΚΟ ΑΛΜΑ", "color": "#DC143C", "symbol": "⤊", "min": 0.75, "max": 0.85},
    6: {"name": "τ₆: ΠΑΡΑΔΟΞΟ", "color": "#8A2BE2", "symbol": "⟡", "min": 0.85, "max": 0.95},
    7: {"name": "τ₇: ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ", "color": "#483D8B", "symbol": "🌀", "min": 0.0, "max": 0.25},
    8: {"name": "τ₈: ΜΕΤΑ-ΣΤΑΘΕΡΟΤΗΤΑ", "color": "#000080", "symbol": "∞", "min": 0.95, "max": 1.5},
    9: {"name": "τ₉: ΜΕΤΑ-ΥΠΕΡΒΑΣΗ", "color": "#000000", "symbol": "🌟", "min": 1.5, "max": float('inf')}
}

# ============================================================================
# ΜΕΡΟΣ 2: XENOPOULOSUNITYFIELD - ΤΟ ΕΝΟΠΟΙΗΜΕΝΟ ΠΕΔΙΟ
# ============================================================================

class XenopoulosUnityField:
    """
    Το Ενοποιημένο Πεδίο του Ξενόπουλου (XenopoulosUnityField)

    Χ(x, t) = L(x) ⊕ ∫₀ᵗ S(τ) dτ

    όπου:
    - L(x): το στιγμιαίο layer (Αρχή 4: Υπέρβαση)
    - S(τ): το ιστορικό σύστημα (Αρχή 6: Αυτο-αναφορικότητα)
    - ⊕: τελεστής διαλεκτικής σύνθεσης (Αρχή 34: Ολοκλήρωση-Υπέρβαση)
    - Ανίχνευση παραδόξων (Αρχή 9: Παραδοξογένεση)
    """

    def __init__(self, w=0.5, r=0.15, horizon=100):
        self.w = w
        self.r = r
        self.horizon = horizon
        self.history = []          # Ιστορία τιμών Α
        self.field_history = []     # Ιστορία τιμών πεδίου
        self.time = 0

    # ------------------------------------------------------------------------
    # 1. ΣΤΙΓΜΙΑΙΟ ΜΕΡΟΣ (Layer) - Αρχή 4: Υπέρβαση
    # ------------------------------------------------------------------------
    def layer(self, x):
        """
        L(x) = tanh(x·w)·(1 - r)

        Το στιγμιαίο μέρος - η άμεση απόκριση στο ερέθισμα.
        """
        return np.tanh(x * self.w) * (1 - self.r)

    # ------------------------------------------------------------------------
    # 2. ΙΣΤΟΡΙΚΟ ΜΕΡΟΣ (System) - Αρχή 6: Αυτο-αναφορικότητα
    # ------------------------------------------------------------------------
    def system(self, t, A):
        """
        S(t) = ¬ᴰ(A) = -A·p·h·(1 + m) + ε

        Το ιστορικό μέρος - επηρεάζεται από το παρελθόν.
        """
        # Memory effect (αυτο-αναφορικότητα)
        if len(self.history) > 0:
            window = min(10, len(self.history))
            recent = np.mean(self.history[-window:])
            memory = 0.2 * np.tanh(recent * 2)
        else:
            memory = 0

        # Στοχαστικοί παράγοντες
        p = 0.7 + 0.3 * random.random()      # preservation
        h = 1.0 + 0.3 * random.random()      # historical weight
        noise = 0.05 * (1 + abs(A)) * random.gauss(0, 1)

        # Διαλεκτική άρνηση
        return -A * p * h * (1 + memory) + noise

    # ------------------------------------------------------------------------
    # 3. ΤΕΛΕΣΤΗΣ ΔΙΑΛΕΚΤΙΚΗΣ ΣΥΝΘΕΣΗΣ (⊕) - Αρχή 34: Ολοκλήρωση-Υπέρβαση
    # ------------------------------------------------------------------------
    def compose(self, a, b):
        """
        a ⊕ b = (a + b)·(1 - |tanh(a·b)|) + i·(a·b)

        Ο τελεστής που ενώνει στιγμιαίο και ιστορικό μέρος.
        Το αποτέλεσμα είναι ΜΙΓΑΔΙΚΟ:
        - Re: η σύνθεση (τι ισχυρίζεται το σύστημα)
        - Im: η ένταση (πόσο "ταράζεται" το σύστημα)
        """
        real_part = (a + b) * (1 - abs(np.tanh(a * b + 1e-10)))
        imag_part = a * b
        return real_part + 1j * imag_part

    # ------------------------------------------------------------------------
    # 4. ΥΠΟΛΟΓΙΣΜΟΣ ΕΝΤΑΣΗΣ
    # ------------------------------------------------------------------------
    def tension(self, X):
        """T = |Im(Χ)| - η διαλεκτική ένταση"""
        return abs(np.imag(X))

    # ------------------------------------------------------------------------
    # 5. ΤΑΞΙΝΟΜΗΣΗ ΣΤΑΔΙΟΥ - Αρχή 9: Παραδοξογένεση
    # ------------------------------------------------------------------------
    def classify_stage(self, X):
        """
        Ταξινόμηση σε στάδιο με βάση:
        - Re(Χ): τι ισχυρίζεται
        - Im(Χ): ένταση
        - Tension: κρυμμένη ένταση
        """
        Re = np.real(X)
        Im = np.imag(X)
        T = self.tension(X)

        # Στάδιο 6: ΠΑΡΑΔΟΞΟ (τ₆)
        if abs(Re) > 0.85 and abs(Im) > 0.85 and T < 0.4:
            return 6, STAGES[6]["name"]

        # Στάδιο 7: ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)
        if T < 0.25 and (abs(Re) > 0.75 or abs(Im) > 0.75):
            return 7, STAGES[7]["name"]

        # Κανονικά στάδια με βάση την ένταση
        for stage_id, stage_info in STAGES.items():
            if stage_id not in [6, 7]:  # Εξαίρεση ειδικών σταδίων
                if stage_info["min"] <= T < stage_info["max"]:
                    return stage_id, stage_info["name"]

        return 0, STAGES[0]["name"]

    # ------------------------------------------------------------------------
    # 6. ΥΠΟΛΟΓΙΣΜΟΣ XEPTQLRI
    # ------------------------------------------------------------------------
    def calculate_xeptqlri(self, X):
        """
        XEPTQLRI = [T^1.2 · trend · paradox · asym] / 0.85
        """
        Re = np.real(X)
        Im = np.imag(X)
        T = self.tension(X)

        # Base
        base = T ** 1.2

        # Trend factor (από ιστορία)
        if len(self.field_history) >= 5:
            recent = [np.imag(f) for f in self.field_history[-5:]]
            if len(recent) > 1:
                trend_coef = np.polyfit(range(len(recent)), recent, 1)[0]
                trend = 1.0 + abs(trend_coef) * 15
            else:
                trend = 1.0
        else:
            trend = 1.0

        # Paradox factor
        paradox = 1.0
        if abs(Re) > 0.85 and abs(Im) > 0.85:
            paradox = 2.8 if T < 0.4 else 2.0

        # Asymmetry factor
        asym = 1.0 + (1 - abs(abs(Re) - abs(Im))) * 0.5

        return min(5.0, max(0.0, (base * trend * paradox * asym) / 0.85))

    # ------------------------------------------------------------------------
    # 7. ΟΛΟΚΛΗΡΩΤΙΚΗ ΜΟΡΦΗ - ΤΟ ΠΛΗΡΕΣ ΠΕΔΙΟ
    # ------------------------------------------------------------------------
    def forward(self, x, A=None):
        """
        Χ(x, t) = L(x) ⊕ ∫₀ᵗ S(τ) dτ

        Υπολογίζει το πεδίο τη χρονική στιγμή t.
        """
        # Αν δεν δόθηκε Α, υπολόγισε από ιστορία
        if A is None:
            A = 0.5 + 0.3 * np.sin(self.time * 0.1) if len(self.history) > 0 else 0.5

        # Στιγμιαίο μέρος
        L = self.layer(x)

        # Ολοκλήρωμα ιστορικού μέρους
        integral = 0
        for tau in range(min(self.time, self.horizon)):
            if tau < len(self.history):
                integral += self.system(tau, self.history[tau])

        # Σύνθεση
        X = self.compose(L, integral)

        # Καταγραφή
        self.field_history.append(X)
        self.history.append(A)
        self.time += 1

        # Υπολογισμός μετρικών
        stage_id, stage_name = self.classify_stage(X)
        xeptqlri = self.calculate_xeptqlri(X)

        return {
            'field': X,
            'real': np.real(X),
            'imag': np.imag(X),
            'tension': self.tension(X),
            'stage_id': stage_id,
            'stage': stage_name,
            'xeptqlri': xeptqlri,
            'layer': L,
            'integral': integral,
            'time': self.time
        }

    # ------------------------------------------------------------------------
    # 8. ΕΝΤΟΠΙΣΜΟΣ ΠΑΡΑΔΟΞΩΝ ΣΕ ΚΕΙΜΕΝΟ
    # ------------------------------------------------------------------------
    def analyze_text(self, text):
        """
        Αναλύει ένα κείμενο για παράδοξα και αντιφάσεις.
        """
        # Χωρισμός σε προτάσεις
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        results = []

        for i, sentence in enumerate(sentences):
            # Υπολογισμός φορτίου πρότασης
            words = sentence.lower().split()

            # Ανίχνευση συναισθηματικού φορτίου
            positive_words = ['ευγνωμοσύνη', 'εμπιστοσύνη', 'χαρά', 'καλός', 'σωστός']
            negative_words = ['μηνύω', 'καταγγέλλω', 'ψέμα', 'λάθος', 'πρόβλημα']

            pos_count = sum(1 for w in words if w in positive_words)
            neg_count = sum(1 for w in words if w in negative_words)

            # Ανίχνευση χρονολογιών
            dates = re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', sentence)

            # Ανίχνευση ποσών
            amounts = re.findall(r'\d+[.,]?\d*\s*[€$]', sentence)

            # Υπολογισμός εισόδου για το πεδίο
            x = (pos_count - neg_count) / max(len(words), 1) + 0.5

            # Εφαρμογή πεδίου
            result = self.forward(x)
            results.append({
                'sentence': sentence[:50] + ('...' if len(sentence) > 50 else ''),
                'index': i,
                'field': result,
                'dates': dates,
                'amounts': amounts,
                'pos_neg': (pos_count, neg_count)
            })

        return results

    # ------------------------------------------------------------------------
    # 9. ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ
    # ------------------------------------------------------------------------
    def statistics(self):
        """Στατιστικά του πεδίου"""
        if not self.field_history:
            return {}

        real_parts = [np.real(f) for f in self.field_history]
        imag_parts = [np.imag(f) for f in self.field_history]

        return {
            'real_mean': float(np.mean(real_parts)),
            'real_std': float(np.std(real_parts)),
            'real_max': float(np.max(real_parts)),
            'real_min': float(np.min(real_parts)),
            'imag_mean': float(np.mean(imag_parts)),
            'imag_std': float(np.std(imag_parts)),
            'imag_max': float(np.max(imag_parts)),
            'imag_min': float(np.min(imag_parts)),
            'energy': float(np.sum([abs(f)**2 for f in self.field_history])),
            'steps': len(self.field_history)
        }

    # ------------------------------------------------------------------------
    # 10. RESET
    # ------------------------------------------------------------------------
    def reset(self):
        """Επαναφορά πεδίου"""
        self.history = []
        self.field_history = []
        self.time = 0


# ============================================================================
# ΜΕΡΟΣ 3: TEXT ANALYZER (Προσαρμοσμένος για UnityField)
# ============================================================================

class TextAnalyzer:
    """Ανάλυση κειμένου με UnityField"""

    def __init__(self):
        self.field = XenopoulosUnityField()
        self.analyses = []

    def lexical_contradictions(self, text):
        """Λεξικές αντιφάσεις (αλλά, όμως, κλπ)"""
        words = ['αλλά', 'όμως', 'παρόλα', 'αντίθετα', 'μολονότι', 'ωστόσο', 'ενώ']
        count = 0
        sentences = [s for s in text.split('.') if s.strip()]

        for i, s in enumerate(sentences):
            found = [w for w in words if w in s.lower()]
            if found:
                count += 1

        return count

    def semantic_analysis(self, text):
        """Νοηματική ανάλυση με UnityField"""
        self.field.reset()
        results = self.field.analyze_text(text)

        # Συγκεντρωτικά στατιστικά
        real_vals = [r['field']['real'] for r in results]
        imag_vals = [r['field']['imag'] for r in results]
        tensions = [r['field']['tension'] for r in results]
        stages = [r['field']['stage_id'] for r in results]

        # Ανίχνευση παραδόξων
        paradoxes = sum(1 for s in stages if s == 6)
        false_stability = sum(1 for s in stages if s == 7)

        return {
            'words': len(text.split()),
            'sentences': len(results),
            'unique_words': len(set(text.lower().split())),
            'lexical_diversity': len(set(text.lower().split())) / max(len(text.split()), 1),
            'avg_sentence_length': len(text.split()) / max(len(results), 1),
            'readability': max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)),
            'lexical_contradictions': self.lexical_contradictions(text),
            'semantic_contradictions': paradoxes + false_stability,
            'paradoxes': paradoxes,
            'false_stability': false_stability,
            'avg_real': float(np.mean(real_vals)) if real_vals else 0,
            'avg_imag': float(np.mean(imag_vals)) if imag_vals else 0,
            'avg_tension': float(np.mean(tensions)) if tensions else 0,
            'max_tension': float(np.max(tensions)) if tensions else 0,
            'score': float(0.3 * (len(set(text.lower().split()))/max(len(text.split()),1)) +
                          0.4 * max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)) +
                          0.3 * (1 - (paradoxes + false_stability)/max(len(results),1))),
            'quality': self._get_quality(0.3 * (len(set(text.lower().split()))/max(len(text.split()),1)) +
                                         0.4 * max(0, min(1, 1 - (len(text.split())/max(len(results),1) - 10)/40)) +
                                         0.3 * (1 - (paradoxes + false_stability)/max(len(results),1)))
        }

    def _get_quality(self, score):
        if score > 0.8: return "ΕΞΑΙΡΕΤΙΚΗ"
        if score > 0.6: return "ΚΑΛΗ"
        if score > 0.4: return "ΜΕΤΡΙΑ"
        return "ΧΑΜΗΛΗ"

    def full_metrics(self, text):
        """Πλήρεις μετρικές"""
        semantic = self.semantic_analysis(text)

        # Υπολογισμός XEPTQLRI από τελευταία πρόταση
        self.field.reset()
        results = self.field.analyze_text(text)
        last_xeptqlri = results[-1]['field']['xeptqlri'] if results else 0

        return {
            'Βασικά': {
                'Χαρακτήρες': len(text),
                'Λέξεις': semantic['words'],
                'Προτάσεις': semantic['sentences']
            },
            'Γλωσσικά': {
                'Λεξιλογική ποικιλία': f"{semantic['lexical_diversity']*100:.1f}%",
                'Μ.Ο. μήκος πρότασης': f"{semantic['avg_sentence_length']:.1f}",
                'Αναγνωσιμότητα': f"{semantic['readability']*100:.1f}%"
            },
            'Αντιφάσεις': {
                'Λεξικές/Συντακτικές': semantic['lexical_contradictions'],
                'Νοηματικές': semantic['semantic_contradictions'],
                'Παράδοξα (τ₆)': semantic['paradoxes'],
                'Ψευδής σταθερότητα (τ₇)': semantic['false_stability'],
                'Σύνολο': semantic['lexical_contradictions'] + semantic['semantic_contradictions']
            },
            'XEPTQLRI': {
                'Δείκτης': f"{last_xeptqlri:.4f}",
                'Μέση ένταση': f"{semantic['avg_tension']:.4f}",
                'Μέγιστη ένταση': f"{semantic['max_tension']:.4f}"
            },
            'Ποιότητα': semantic['quality']
        }

    def final_assessment(self, text):
        """Τελική αποτίμηση με UnityField"""
        semantic = self.semantic_analysis(text)

        result = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║              ΤΕΛΙΚΗ ΑΠΟΤΙΜΗΣΗ ΜΕ XENOPOULOSUNITYFIELD                    ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  ⚠️  ΠΙΝΑΚΑΣ ΑΝΤΙΦΑΣΕΩΝ                                                  ║
║  ────────────────────────────────────────────────────────────────────── ║
║                                                                          ║
║  Τύπος Αντίφασης                  |  Πλήθος  |  Σοβαρότητα              ║
║  ────────────────────────────────────────────────────────────────────── ║
║  Λεξικές/Συντακτικές                |  {semantic['lexical_contradictions']:4d}      |  25%                       ║
║  Νοηματικές (σύνολο)                |  {semantic['semantic_contradictions']:4d}      |  Ποικίλη                   ║
║    • Παράδοξα (τ₆)                   |  {semantic['paradoxes']:4d}      |  85% (ΚΡΙΣΙΜΟ)              ║
║    • Ψευδής Σταθερότητα (τ₇)         |  {semantic['false_stability']:4d}      |  70% (ΥΠΟΠΤΟ)               ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  🔴 ΑΝΑΛΥΣΗ ΠΕΔΙΟΥ:                                                      ║
║     • Μέσο Re(Χ): {semantic['avg_real']:.4f} (τι ισχυρίζεται)                           ║
║     • Μέσο Im(Χ): {semantic['avg_imag']:.4f} (διαλεκτική ένταση)                        ║
║     • Μέση ένταση: {semantic['avg_tension']:.4f}                                        ║
║     • Μέγιστη ένταση: {semantic['max_tension']:.4f}                                      ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  🔍 ΕΝΤΟΠΙΣΜΕΝΑ ΠΑΡΑΔΟΞΑ:                                                ║
"""

        if semantic['paradoxes'] > 0:
            result += f"""
║     • Βρέθηκαν {semantic['paradoxes']} ΠΑΡΑΔΟΞΑ (τ₆) - ΑΜΕΣΗ ΠΡΟΣΟΧΗ!    ║
║     • Το κείμενο περιέχει ΛΟΓΙΚΕΣ ΑΝΤΙΦΑΣΕΙΣ                             ║
"""
        else:
            result += """
║     • Δεν βρέθηκαν παράδοξα (τ₆) - ΚΑΛΟ!                                 ║
"""

        if semantic['false_stability'] > 0:
            result += f"""
║     • Βρέθηκαν {semantic['false_stability']} περιπτώσεις ΨΕΥΔΟΥΣ ΣΤΑΘΕΡΟΤΗΤΑΣ (τ₇)   ║
║     • Υπάρχουν ΚΡΥΦΕΣ αντιφάσεις - ΠΡΟΣΟΧΗ!                              ║
"""

        result += f"""
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  💡 ΣΧΕΔΙΟ ΔΡΑΣΗΣ:                                                       ║
║                                                                          ║
║  ΦΑΣΗ 1 - Άμεσες ενέργειες (Προτεραιότητα ★★★):                         ║
"""

        if semantic['paradoxes'] > 0:
            result += f"""
║     • Διόρθωσε {semantic['paradoxes']} ΠΑΡΑΔΟΞΑ (λογικές αντιφάσεις)     ║
"""
        else:
            result += """
║     • Καμία άμεση ενέργεια για παράδοξα - ΠΡΟΧΩΡΑ                         ║
"""

        result += f"""
║                                                                          ║
║  ΦΑΣΗ 2 - Βελτιστοποίηση (Προτεραιότητα ✦):                             ║
║     • Διόρθωσε {semantic['lexical_contradictions']} λεξικές/συντακτικές αντιφάσεις   ║
║     • Διόρθωσε {semantic['false_stability']} περιπτώσεις ψευδούς σταθερότητας       ║
║                                                                          ║
║  ΦΑΣΗ 3 - Τελικός έλεγχος:                                               ║
║     • Επαναξιολόγηση με UnityField                                      ║
║     • Σύγκριση αρχικού-διορθωμένου                                       ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
        return result

    def unity_field_report(self, text):
        """Αναλυτική αναφορά UnityField"""
        self.field.reset()
        results = self.field.analyze_text(text)

        report = f"""
🌀 XENOPOULOS UNITY FIELD - ΑΝΑΛΥΤΙΚΗ ΑΝΑΦΟΡΑ
================================================================================

📊 ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ:
   • Βήματα: {len(results)}
   • Τελικό Re(Χ): {results[-1]['field']['real']:.4f}
   • Τελικό Im(Χ): {results[-1]['field']['imag']:.4f}
   • Τελική ένταση: {results[-1]['field']['tension']:.4f}
   • Τελικό στάδιο: {results[-1]['field']['stage']}
   • Τελικό XEPTQLRI: {results[-1]['field']['xeptqlri']:.4f}

🔍 ΑΝΑΛΥΣΗ ΑΝΑ ΠΡΟΤΑΣΗ:
"""
        for i, r in enumerate(results[:10]):  # Πρώτες 10 προτάσεις
            report += f"""
   {i+1}. {r['sentence']}
       • Re={r['field']['real']:.3f}, Im={r['field']['imag']:.3f}, T={r['field']['tension']:.3f}
       • Στάδιο: {r['field']['stage']}
       • Ημερομηνίες: {r['dates']}, Ποσά: {r['amounts']}
"""

        if len(results) > 10:
            report += f"\n   ... και {len(results)-10} ακόμη προτάσεις"

        stats = self.field.statistics()
        report += f"""

📈 ΣΥΝΟΛΙΚΑ ΣΤΑΤΙΣΤΙΚΑ ΠΕΔΙΟΥ:
   • Μέσο Re(Χ): {stats.get('real_mean', 0):.4f}
   • Μέσο Im(Χ): {stats.get('imag_mean', 0):.4f}
   • Μέγιστη ένταση: {stats.get('imag_max', 0):.4f}
   • Ολική ενέργεια: {stats.get('energy', 0):.4f}
   • Βήματα: {stats.get('steps', 0)}

🏆 ΤΕΛΙΚΗ ΔΙΑΓΝΩΣΗ:
"""
        last = results[-1]['field']
        if last['stage_id'] == 6:
            report += "🔴 ΤΟ ΚΕΙΜΕΝΟ ΠΕΡΙΕΧΕΙ ΠΑΡΑΔΟΞΑ - ΑΠΑΙΤΕΙΤΑΙ ΔΙΟΡΘΩΣΗ!"
        elif last['stage_id'] == 7:
            report += "⚠️ ΤΟ ΚΕΙΜΕΝΟ ΕΧΕΙ ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ - ΥΠΑΡΧΟΥΝ ΚΡΥΦΕΣ ΑΝΤΙΦΑΣΕΙΣ"
        elif last['tension'] < 0.3:
            report += "✅ ΤΟ ΚΕΙΜΕΝΟ ΕΙΝΑΙ ΛΟΓΙΚΑ ΣΥΝΕΠΕΣ"
        else:
            report += f"📊 ΤΟ ΚΕΙΜΕΝΟ ΒΡΙΣΚΕΤΑΙ ΣΕ ΚΑΤΑΣΤΑΣΗ: {last['stage']}"

        return report


# ============================================================================
# ΜΕΡΟΣ 4: ΑΣΦΑΛΕΙΑ
# ============================================================================

class SecuritySystem:
    def __init__(self):
        self.level = 'user'
        self.keys = ['15081920', '15-08-1920', '15 Αυγούστου 1920']

    def authenticate(self, key):
        if key in self.keys:
            self.level = 'admin'
            return True
        return False


# ============================================================================
# ΜΕΡΟΣ 5: MEMORY MANAGER
# ============================================================================

class MemoryManager:
    def __init__(self, memory_file="exdt_memory.json"):
        self.memory_file = memory_file

    def save(self, data):
        with open(self.memory_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False, default=str)
        return True

    def load(self):
        if os.path.exists(self.memory_file):
            with open(self.memory_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        return None


# ============================================================================
# ΜΕΡΟΣ 6: ΠΛΗΡΕΣ ΣΥΣΤΗΜΑ
# ============================================================================

class XenopoulosSystem:
    def __init__(self, memory_file="exdt_memory.json"):
        self.text = TextAnalyzer()
        self.unity_field = XenopoulosUnityField()
        self.security = SecuritySystem()
        self.memory = MemoryManager(memory_file)
        self.analyses = []
        self.load_memory()

    def load_memory(self):
        data = self.memory.load()
        if data:
            self.analyses = data.get('analyses', [])
            print(f"📂 Φορτώθηκαν {len(self.analyses)} προηγούμενες αναλύσεις")

    def save_memory(self):
        data = {'analyses': self.analyses, 'last_save': datetime.now().isoformat()}
        self.memory.save(data)

    def analyze_text(self, text, save=True):
        result = {
            'id': str(uuid.uuid4())[:8],
            'timestamp': datetime.now().isoformat(),
            'text_preview': text[:100] + ('...' if len(text) > 100 else ''),
            'semantic': self.text.semantic_analysis(text),
            'unity': self.text.unity_field_report(text)
        }

        if save:
            self.analyses.append(result)
            if len(self.analyses) % 5 == 0:
                self.save_memory()

        return result

    def info(self):
        return {
            'version': VERSION,
            'access': self.security.level,
            'analyses': len(self.analyses)
        }


# ============================================================================
# ΜΕΡΟΣ 7: UI
# ============================================================================

class XenopoulosUI:
    def __init__(self, system):
        self.sys = system
        self.text = system.text
        if IPYWIDGETS_AVAILABLE:
            self._build()

    def _build(self):
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #667eea, #764ba2);
             padding: 20px; border-radius: 15px; color: white; text-align: center;">
            <h1>🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ</h1>
            <h2>EXDT v3.0 με XENOPOULOSUNITYFIELD</h2>
            <p>9 Λειτουργίες Ανάλυσης | 34 Αρχές | 7 Θεωρήματα</p>
        </div>
        """))

        self.input = widgets.Textarea(
            value='Επικολλήστε το κείμενό σας εδώ...',
            layout=widgets.Layout(width='100%', height='150px')
        )
        display(self.input)

        # 9 κουμπιά
        btns = []
        btn_configs = [
            ('1️⃣ ΛΕΞΙΚΕΣ', 'primary', self._lexical),
            ('2️⃣ ΝΟΗΜΑΤΙΚΗ', 'success', self._semantic),
            ('3️⃣ ΔΙΟΡΘΩΣΗ', 'info', self._correct),
            ('4️⃣ ΛΙΣΤΑ', 'warning', self._list),
            ('5️⃣ ΜΕΤΡΙΚΕΣ', 'danger', self._metrics),
            ('6️⃣ ΣΥΝΟΛΙΚΗ', 'primary', self._report),
            ('7️⃣ ΤΕΛΙΚΗ v3', 'success', self._final),
            ('8️⃣ EXTREME', 'danger', self._extreme),
            ('0️⃣ UNITY FIELD', 'info', self._unity)
        ]

        for txt, style, handler in btn_configs:
            btn = widgets.Button(description=txt, button_style=style,
                               layout=widgets.Layout(width='160px', margin='2px'))
            btn.on_click(handler)
            btns.append(btn)

        display(widgets.HBox(btns[:5]))
        display(widgets.HBox(btns[5:]))

        # Output
        self.out = widgets.Output(layout=widgets.Layout(width='100%', height='500px', overflow='auto'))
        display(self.out)

    def _get_text(self):
        t = self.input.value
        if t in ['Επικολλήστε το κείμενό σας εδώ...', '']:
            with self.out:
                clear_output()
                print("❌ Παρακαλώ εισάγετε κείμενο")
            return None
        return t

    def _lexical(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            c = self.text.lexical_contradictions(t)
            print(f"🔍 ΛΕΞΙΚΕΣ ΑΝΤΙΦΑΣΕΙΣ: {c}")

    def _semantic(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            m = self.text.semantic_analysis(t)
            print("📊 ΝΟΗΜΑΤΙΚΗ ΑΝΑΛΥΣΗ\n" + "="*50)
            for k, v in m.items():
                if k not in ['avg_real', 'avg_imag', 'avg_tension', 'max_tension', 'paradoxes', 'false_stability']:
                    print(f"{k}: {v}")
            print(f"\n📈 ΣΤΑΤΙΣΤΙΚΑ UNITY FIELD:")
            print(f"   Μέσο Re(Χ): {m['avg_real']:.4f}")
            print(f"   Μέσο Im(Χ): {m['avg_imag']:.4f}")
            print(f"   Μέση ένταση: {m['avg_tension']:.4f}")
            print(f"   Μέγιστη ένταση: {m['max_tension']:.4f}")
            print(f"   Παράδοξα (τ₆): {m['paradoxes']}")
            print(f"   Ψευδής σταθερότητα (τ₇): {m['false_stability']}")

    def _correct(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print("✏️ ΔΙΟΡΘΩΣΗ\n" + "="*50)
            print("Η λειτουργία διόρθωσης θα ενσωματωθεί σύντομα")

    def _list(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            self.text.field.reset()
            results = self.text.field.analyze_text(t)
            print("📋 ΛΕΠΤΟΜΕΡΗΣ ΛΙΣΤΑ ΑΝΑΛΥΣΗΣ\n" + "="*50)
            for i, r in enumerate(results[:10]):
                print(f"\n{i+1}. {r['sentence']}")
                print(f"   Re={r['field']['real']:.3f}, Im={r['field']['imag']:.3f}")
                print(f"   Στάδιο: {r['field']['stage']}")
                if r['dates']:
                    print(f"   Ημερομηνίες: {r['dates']}")
                if r['amounts']:
                    print(f"   Ποσά: {r['amounts']}")

    def _metrics(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            m = self.text.full_metrics(t)
            print("📊 ΠΛΗΡΕΙΣ ΜΕΤΡΙΚΕΣ\n" + "="*50)
            for category, values in m.items():
                print(f"\n{category}:")
                if isinstance(values, dict):
                    for k, v in values.items():
                        print(f"  {k}: {v}")
                else:
                    print(f"  {values}")

    def _report(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.final_assessment(t))

    def _final(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.final_assessment(t))

    def _extreme(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            self.text.field.reset()
            results = self.text.field.analyze_text(t)
            last = results[-1]['field']
            print("🔥 ΑΝΑΛΥΣΗ EXTREME ΚΑΤΑΣΤΑΣΗΣ\n" + "="*50)
            print(f"XEPTQLRI: {last['xeptqlri']:.4f}")
            if last['xeptqlri'] >= 2.0:
                print("🚨 EXTREME ΚΑΤΑΣΤΑΣΗ! (XEPTQLRI > 2.0)")
            elif last['xeptqlri'] >= 1.5:
                print("⚠️ ΥΠΕΡΚΡΙΣΙΜΗ ΚΑΤΑΣΤΑΣΗ")
            elif last['stage_id'] == 6:
                print("🔴 ΠΑΡΑΔΟΞΟ (τ₆)")
            elif last['stage_id'] == 7:
                print("⚠️ ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)")
            else:
                print("✅ ΦΥΣΙΟΛΟΓΙΚΗ ΚΑΤΑΣΤΑΣΗ")

    def _unity(self, b):
        with self.out:
            clear_output()
            t = self._get_text()
            if not t: return
            print(self.text.unity_field_report(t))


# ============================================================================
# ΜΕΡΟΣ 8: ΕΚΚΙΝΗΣΗ
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ - ΔΙΑΛΕΚΤΙΚΟΣ ΜΕΤΑΣΧΗΜΑΤΙΣΤΗΣ")
    print("="*80)
    print("📌 34 Αρχές | 7 Θεωρήματα | 9 Λειτουργίες | XENOPOULOSUNITYFIELD")
    print("📅 Ημερομηνία γέννησης: 15 Αυγούστου 1920")
    print("🔑 Κλειδί: 15081920 | 15-08-1920 | 15 Αυγούστου 1920")
    print("="*80)

    system = XenopoulosSystem()

    if IPYWIDGETS_AVAILABLE:
        ui = XenopoulosUI(system)
        print("\n✅ Σύστημα έτοιμο - 9 λειτουργίες (ΝΕΟ: 0️⃣ UNITY FIELD)")
    else:
        print("⚠️ Εκτέλεση σε console mode")

    print("\n" + "="*80)
    print("🚀 ΝΕΑ ΛΕΙΤΟΥΡΓΙΑ: XENOPOULOSUNITYFIELD")
    print("   • Ενοποιεί Layer (στιγμιαίο) και System (ιστορικό)")
    print("   • Ανιχνεύει ΠΑΡΑΔΟΞΑ (τ₆) και ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)")
    print("   • Χρησιμοποιεί μιγαδική ανάλυση (Re + i·Im)")
    print("="*80)

    # ============================================================================
# ΑΥΤΟΜΑΤΗ ΑΠΟΘΗΚΕΥΣΗ ΟΛΩΝ ΤΩΝ ΑΠΟΤΕΛΕΣΜΑΤΩΝ ΚΑΙ ΓΡΑΦΗΜΑΤΩΝ
# ============================================================================

import json
import pickle
from datetime import datetime
import os
import matplotlib.pyplot as plt
import numpy as np

class EXDTAutoSave:
    """
    Αυτόματη αποθήκευση ΟΛΩΝ των αποτελεσμάτων και γραφημάτων
    """

    def __init__(self, session_name=None):
        # Δημιουργία μοναδικού ονόματος session
        if session_name is None:
            self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        else:
            self.session_id = session_name

        # Δημιουργία φακέλων
        self.base_dir = f"EXDT_Session_{self.session_id}"
        self.text_dir = f"{self.base_dir}/text_reports"
        self.graphs_dir = f"{self.base_dir}/graphs"
        self.json_dir = f"{self.base_dir}/json_data"
        self.summary_dir = f"{self.base_dir}/summary"

        for dir_path in [self.text_dir, self.graphs_dir, self.json_dir, self.summary_dir]:
            os.makedirs(dir_path, exist_ok=True)

        # Αρχείο καταγραφής session
        self.log_file = f"{self.base_dir}/session_log.txt"
        self._log(f"🔥 NEW EXDT SESSION: {self.session_id}")
        self._log(f"📁 Results saved in: {self.base_dir}")

        # Συλλογή όλων των αποτελεσμάτων
        self.all_unity_reports = []
        self.all_metrics = []
        self.all_extreme = []
        self.graph_count = 0

    def _log(self, message):
        """Εσωτερική καταγραφή"""
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(f"{datetime.now().strftime('%H:%M:%S')} - {message}\n")
        print(message)

    # ------------------------------------------------------------------------
    # 1. ΑΠΟΘΗΚΕΥΣΗ ΓΡΑΦΗΜΑΤΩΝ
    # ------------------------------------------------------------------------

    def save_current_figure(self, name, fig=None, dpi=300, formats=['png', 'pdf', 'svg']):
        """
        Αποθήκευση του τρέχοντος γραφήματος σε πολλαπλές μορφές
        """
        if fig is None:
            fig = plt.gcf()

        timestamp = datetime.now().strftime("%H%M%S")
        self.graph_count += 1

        saved_files = []
        for fmt in formats:
            filename = f"{self.graphs_dir}/{self.graph_count:03d}_{name}_{timestamp}.{fmt}"
            fig.savefig(filename, dpi=dpi, bbox_inches='tight', format=fmt)
            saved_files.append(filename)

        self._log(f"📊 Γράφημα {self.graph_count}: {name} αποθηκεύτηκε σε {len(saved_files)} μορφές")
        return saved_files

    def save_unity_field_plots(self, results, name="unity_field"):
        """
        Αποθήκευση όλων των γραφημάτων του Unity Field
        """
        if not results:
            return

        # Δημιουργία figure με 4 υπογραφήματα
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'XenopoulosUnityField - {name}', fontsize=16)

        steps = range(len(results))
        real_parts = [r['field']['real'] for r in results]
        imag_parts = [r['field']['imag'] for r in results]
        tensions = [r['field']['tension'] for r in results]

        # 1. Re(Χ) και Im(Χ)
        ax1 = axes[0, 0]
        ax1.plot(steps, real_parts, 'b-', label='Re(Χ)', linewidth=2)
        ax1.plot(steps, imag_parts, 'r--', label='Im(Χ)', linewidth=2)
        ax1.axhline(y=0.85, color='purple', linestyle=':', label='Όριο παραδόξου')
        ax1.set_xlabel('Χρονικό Βήμα')
        ax1.set_ylabel('Τιμή')
        ax1.set_title('Πραγματικό και Φανταστικό Μέρος')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2. Διαλεκτική Ένταση
        ax2 = axes[0, 1]
        ax2.plot(steps, tensions, 'purple', linewidth=2)
        ax2.axhline(y=0.4, color='r', linestyle='--', label='Όριο παραδόξου')
        ax2.axhline(y=0.25, color='orange', linestyle='--', label='Όριο ψευδούς σταθ.')
        ax2.fill_between(steps, 0, tensions, alpha=0.3, color='purple')
        ax2.set_xlabel('Χρονικό Βήμα')
        ax2.set_ylabel('Ένταση |Im(Χ)|')
        ax2.set_title('Διαλεκτική Ένταση')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3. Φασικό διάγραμμα
        ax3 = axes[1, 0]
        scatter = ax3.scatter(real_parts, imag_parts, c=steps, cmap='viridis',
                            alpha=0.7, s=50)
        plt.colorbar(scatter, ax=ax3, label='Χρονικό Βήμα')
        ax3.axhline(y=0.85, color='r', linestyle='--', alpha=0.5)
        ax3.axvline(x=0.85, color='r', linestyle='--', alpha=0.5)
        ax3.set_xlabel('Re(Χ)')
        ax3.set_ylabel('Im(Χ)')
        ax3.set_title('Φασικό Διάγραμμα')
        ax3.grid(True, alpha=0.3)

        # 4. Ιστόγραμμα
        ax4 = axes[1, 1]
        ax4.hist(tensions, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
        ax4.axvline(x=0.4, color='r', linestyle='--', label='Όριο παραδόξου')
        ax4.axvline(x=0.25, color='orange', linestyle='--', label='Όριο ψευδούς')
        ax4.set_xlabel('Διαλεκτική Ένταση')
        ax4.set_ylabel('Συχνότητα')
        ax4.set_title('Κατανομή Εντάσεων')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()

        # Αποθήκευση
        return self.save_current_figure(f"unity_field_{name}", fig)

    # ------------------------------------------------------------------------
    # 2. ΑΠΟΘΗΚΕΥΣΗ ΑΝΑΦΟΡΩΝ
    # ------------------------------------------------------------------------

    def save_unity_report(self, report_text, name="unity_report"):
        """Αποθήκευση αναφοράς Unity Field"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.text_dir}/{name}_{timestamp}.txt"

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_text)

        self._log(f"📝 Αναφορά αποθηκεύτηκε: {filename}")

        # Επίσης αποθήκευση σε JSON
        self.save_json({
            'type': 'unity_report',
            'timestamp': datetime.now().isoformat(),
            'report': report_text,
            'report_length': len(report_text)
        }, f"unity_report_{timestamp}")

        self.all_unity_reports.append({
            'filename': filename,
            'timestamp': datetime.now().isoformat()
        })

        return filename

    def save_metrics(self, metrics_dict, name="metrics"):
        """Αποθήκευση μετρικών"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.json_dir}/{name}_{timestamp}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(metrics_dict, f, indent=2, ensure_ascii=False, default=str)

        self._log(f"📊 Μετρικές αποθηκεύτηκαν: {filename}")
        self.all_metrics.append(metrics_dict)

        # Δημιουργία και σύνοψης
        self._create_metrics_summary(metrics_dict, timestamp)

        return filename

    def save_extreme_analysis(self, result, name="extreme"):
        """Αποθήκευση extreme analysis"""

        timestamp = datetime.now().strftime("%H%M%S")

        # Κείμενο
        text_file = f"{self.text_dir}/{name}_{timestamp}.txt"
        with open(text_file, 'w', encoding='utf-8') as f:
            if 'xeptqlri' in result:
                f.write(f"XEPTQLRI: {result['xeptqlri']}\n")
                if result['xeptqlri'] >= 2.0:
                    f.write("🚨 EXTREME ΚΑΤΑΣΤΑΣΗ!\n")

        # JSON
        json_file = f"{self.json_dir}/{name}_{timestamp}.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, default=str)

        self._log(f"🔥 Extreme analysis: {text_file}")
        self.all_extreme.append(result)

        return text_file

    def save_json(self, data, name="data"):
        """Γενική αποθήκευση JSON"""

        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{self.json_dir}/{name}_{timestamp}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False, default=str)

        return filename

    # ------------------------------------------------------------------------
    # 3. ΣΥΝΟΨΕΙΣ ΚΑΙ ΣΤΑΤΙΣΤΙΚΑ
    # ------------------------------------------------------------------------

    def _create_metrics_summary(self, metrics, timestamp):
        """Δημιουργία σύνοψης από μετρικές"""

        summary_file = f"{self.summary_dir}/summary_{timestamp}.txt"

        with open(summary_file, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write("📊 ΣΥΝΟΨΗ ΜΕΤΡΙΚΩΝ\n")
            f.write("="*60 + "\n\n")

            if 'Βασικά' in metrics:
                f.write("📌 ΒΑΣΙΚΑ ΣΤΟΙΧΕΙΑ:\n")
                for k, v in metrics['Βασικά'].items():
                    f.write(f"   {k}: {v}\n")
                f.write("\n")

            if 'XEPTQLRI' in metrics:
                f.write("🔥 XEPTQLRI:\n")
                for k, v in metrics['XEPTQLRI'].items():
                    f.write(f"   {k}: {v}\n")
                f.write("\n")

            if 'Ποιότητα' in metrics:
                f.write(f"⭐ ΠΟΙΟΤΗΤΑ: {metrics['Ποιότητα']}\n")

        self._log(f"📋 Σύνοψη δημιουργήθηκε: {summary_file}")

    def create_final_report(self):
        """Δημιουργία τελικής αναφοράς όλης της συνεδρίας"""

        report_file = f"{self.base_dir}/FINAL_REPORT.txt"

        with open(report_file, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("🏛️ EXDT v3.0 - ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ ΣΥΝΕΔΡΙΑΣ\n")
            f.write(f"📅 Session ID: {self.session_id}\n")
            f.write(f"🕒 Ημερομηνία: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")

            f.write("📊 ΣΤΑΤΙΣΤΙΚΑ ΣΥΝΕΔΡΙΑΣ:\n")
            f.write(f"   • Unity Reports: {len(self.all_unity_reports)}\n")
            f.write(f"   • Μετρικές: {len(self.all_metrics)}\n")
            f.write(f"   • Extreme Analyses: {len(self.all_extreme)}\n")
            f.write(f"   • Γραφήματα: {self.graph_count}\n\n")

            if self.all_metrics:
                f.write("🔥 ΜΕΓΙΣΤΕΣ ΕΝΤΑΣΕΙΣ:\n")
                for i, m in enumerate(self.all_metrics[-5:]):  # Τελευταίες 5
                    if 'XEPTQLRI' in m:
                        intensity = m['XEPTQLRI'].get('Μέγιστη ένταση', 'N/A')
                        f.write(f"   • Ανάλυση {i+1}: {intensity}\n")

            f.write("\n" + "="*80 + "\n")
            f.write("📁 ΟΛΑ ΤΑ ΑΡΧΕΙΑ ΑΠΟΘΗΚΕΥΤΗΚΑΝ\n")
            f.write(f"📂 Φάκελος: {self.base_dir}\n")

        self._log(f"\n🎉 ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ: {report_file}")
        self._log(f"📁 ΟΛΑ ΤΑ ΑΡΧΕΙΑ ΣΤΟΝ ΦΑΚΕΛΟ: {self.base_dir}")

        return report_file

    # ------------------------------------------------------------------------
    # 4. ΒΟΗΘΗΤΙΚΕΣ ΣΥΝΑΡΤΗΣΕΙΣ
    # ------------------------------------------------------------------------

    def zip_session(self):
        """Δημιουργία zip αρχείου με όλη τη συνεδρία"""

        import shutil

        zip_filename = f"{self.base_dir}.zip"
        shutil.make_archive(self.base_dir, 'zip', self.base_dir)

        self._log(f"📦 Συνεδρία συμπιέστηκε: {zip_filename}")
        return zip_filename

    def download_session(self):
        """Αυτόματο download του zip στο Colab"""

        try:
            from google.colab import files
            zip_file = self.zip_session()
            files.download(zip_file)
            self._log("📥 Το αρχείο κατέβηκε στον υπολογιστή σου!")
        except:
            self._log("⚠️ Δεν μπορεί να γίνει αυτόματο download (μη-Colab περιβάλλον)")


# ============================================================================
# ΠΑΡΑΔΕΙΓΜΑ ΧΡΗΣΗΣ
# ============================================================================

# Δημιουργία αυτόματης αποθήκευσης
saver = EXDTAutoSave("DeepSeek_Analysis_19_3_2026")

# ΜΕΤΑ ΑΠΟ ΚΑΘΕ ΑΝΑΛΥΣΗ:

# 1. Για αποθήκευση αναφοράς Unity Field:
# saver.save_unity_report(unity_report_text, "deepseek_analysis_1")

# 2. Για αποθήκευση γραφημάτων:
# saver.save_unity_field_plots(results, "deepseek_analysis")

# 3. Για αποθήκευση μετρικών:
# saver.save_metrics(metrics_dict, "deepseek_metrics")

# 4. ΣΤΟ ΤΕΛΟΣ:
# saver.create_final_report()
# saver.download_session()  # Κατεβάζει όλα τα αρχεία!


🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ - ΔΙΑΛΕΚΤΙΚΟΣ ΜΕΤΑΣΧΗΜΑΤΙΣΤΗΣ
📌 34 Αρχές | 7 Θεωρήματα | 9 Λειτουργίες | XENOPOULOSUNITYFIELD
📅 Ημερομηνία γέννησης: 15 Αυγούστου 1920
🔑 Κλειδί: 15081920 | 15-08-1920 | 15 Αυγούστου 1920


Textarea(value='Επικολλήστε το κείμενό σας εδώ...', layout=Layout(height='150px', width='100%'))

Output(layout=Layout(height='500px', overflow='auto', width='100%'))


✅ Σύστημα έτοιμο - 9 λειτουργίες (ΝΕΟ: 0️⃣ UNITY FIELD)

🚀 ΝΕΑ ΛΕΙΤΟΥΡΓΙΑ: XENOPOULOSUNITYFIELD
   • Ενοποιεί Layer (στιγμιαίο) και System (ιστορικό)
   • Ανιχνεύει ΠΑΡΑΔΟΞΑ (τ₆) και ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)
   • Χρησιμοποιεί μιγαδική ανάλυση (Re + i·Im)
🔥 NEW EXDT SESSION: DeepSeek_Analysis_19_3_2026
📁 Results saved in: EXDT_Session_DeepSeek_Analysis_19_3_2026

🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ - ΔΙΑΛΕΚΤΙΚΟΣ ΜΕΤΑΣΧΗΜΑΤΙΣΤΗΣ
📌 34 Αρχές | 7 Θεωρήματα | 9 Λειτουργίες | XENOPOULOSUNITYFIELD
📅 Ημερομηνία γέννησης: 15 Αυγούστου 1920
🔑 Κλειδί: 15081920 | 15-08-1920 | 15 Αυγούστου 1920


Textarea(value='Επικολλήστε το κείμενό σας εδώ...', layout=Layout(height='150px', width='100%'))

Output(layout=Layout(height='500px', overflow='auto', width='100%'))


✅ Σύστημα έτοιμο - 9 λειτουργίες (ΝΕΟ: 0️⃣ UNITY FIELD)

🚀 ΝΕΑ ΛΕΙΤΟΥΡΓΙΑ: XENOPOULOSUNITYFIELD
   • Ενοποιεί Layer (στιγμιαίο) και System (ιστορικό)
   • Ανιχνεύει ΠΑΡΑΔΟΞΑ (τ₆) και ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)
   • Χρησιμοποιεί μιγαδική ανάλυση (Re + i·Im)
🔥 NEW EXDT SESSION: DeepSeek_Analysis_19_3_2026
📁 Results saved in: EXDT_Session_DeepSeek_Analysis_19_3_2026

🏛️ ΕΠΑΜΕΙΝΩΝΔΑΣ ΞΕΝΟΠΟΥΛΟΣ - ΔΙΑΛΕΚΤΙΚΟΣ ΜΕΤΑΣΧΗΜΑΤΙΣΤΗΣ
📌 34 Αρχές | 7 Θεωρήματα | 9 Λειτουργίες | XENOPOULOSUNITYFIELD
📅 Ημερομηνία γέννησης: 15 Αυγούστου 1920
🔑 Κλειδί: 15081920 | 15-08-1920 | 15 Αυγούστου 1920


Textarea(value='Επικολλήστε το κείμενό σας εδώ...', layout=Layout(height='150px', width='100%'))

Output(layout=Layout(height='500px', overflow='auto', width='100%'))


✅ Σύστημα έτοιμο - 9 λειτουργίες (ΝΕΟ: 0️⃣ UNITY FIELD)

🚀 ΝΕΑ ΛΕΙΤΟΥΡΓΙΑ: XENOPOULOSUNITYFIELD
   • Ενοποιεί Layer (στιγμιαίο) και System (ιστορικό)
   • Ανιχνεύει ΠΑΡΑΔΟΞΑ (τ₆) και ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ (τ₇)
   • Χρησιμοποιεί μιγαδική ανάλυση (Re + i·Im)
🔥 NEW EXDT SESSION: DeepSeek_Analysis_19_3_2026
📁 Results saved in: EXDT_Session_DeepSeek_Analysis_19_3_2026
